# Лабораторная 3 · Приближённый поиск: чем платят за скорость

**Неделя 10 · занятие 1.** Опора — L13 «ANN: HNSW, IVF, PQ · FAISS · Векторные БД · Продакшн».

До сих пор мы искали перебором: умножали матрицу запросов на матрицу корпуса. На полутора
тысячах документов это микросекунды. На миллиарде — нет. Сегодня строим индексы, которые
ищут **приближённо**, и главный вопрос занятия не «насколько быстрее», а **чем именно
за это платят и как эту плату правильно измерить**.

| # | вопрос занятия | чем отвечаем |
|---|---|---|
| 1 | Когда приближённый поиск вообще окупается? | меряем точный на четырёх размерах корпуса |
| 2 | Чем платят за скорость? | согласие с точным поиском против потери **конечной** метрики |
| 3 | Сколько памяти можно сэкономить? | арифметика PQ и обвал качества при сжатии |

**Данные.** 20 Newsgroups целиком — около девяти тысяч документов после фильтрации, те же
псевдозапросы, что на неделях 7 и 9. Плюс игрушки из `data/l9-*.json` для сверки с доской.

**Среда.** Colab T4 через VS Code. `faiss-cpu`, кодирование корпуса около полуминуты. GPU
не нужен: сегодня всё считается на процессоре, и это принципиально — приближённый поиск
придуман ровно для того, чтобы не требовать ускорителя на каждый запрос.

**Бюджет: ≈120 минут.**

**Артефакт на вынос.** `artifacts/ann.json` — конфигурации индексов с их полнотой, задержкой
и памятью, а также выбранная рабочая точка. На неделе 12 RAG будет строиться поверх неё.

**Как запускать.** Сверху вниз. Эмбеддинги недели 7 подхватываются, если есть, но корпус
сегодня **больше**: на полутора тысячах векторов приближённый поиск смысла не имеет, и мы
это покажем числом.

<details><summary>Почему точный поиск вообще перестаёт работать — арифметика, а не мнение</summary>

Точный поиск ближайших соседей — это умножение матрицы `Q × Eᵀ`. Стоимость линейна
по числу документов и по размерности, и оба множителя в проде большие.

**Считаем.** Миллиард векторов по 768 измерений во float32 — это 3 килобайта на вектор,
то есть **3 терабайта** только на хранение. Уже здесь разговор о «просто умножить матрицы»
заканчивается: такой массив не помещается в память ни одной разумной машины, и каждый запрос
означал бы чтение трёх терабайт с диска.

**Даже если бы поместилось.** Одно скалярное произведение — 768 умножений и сложений.
Миллиард документов — 768 миллиардов операций на запрос. Современный процессор с векторными
инструкциями выдаёт порядка сотни гигафлопс на ядро; получается около восьми секунд на запрос
на ядро. Тысяча запросов в секунду потребовала бы восьми тысяч ядер. Для одного поискового
индекса.

**Что делает приближённый поиск.** Он отказывается от гарантии найти **точных** ближайших
соседей и взамен трогает не весь корпус, а малую его часть. Три семейства, которые мы сегодня
разберём: граф малого мира (HNSW) идёт к ответу по рёбрам, инвертированный файл (IVF) режет
пространство на ячейки и смотрит только ближайшие, квантование произведения (PQ) сжимает
сами векторы, чтобы их стало возможно держать в памяти.

**Главное, что стоит понимать заранее.** Все три метода дают **не тот же** ответ, что точный
поиск. Вопрос не в том, есть ли ошибка, а в том, где она и сколько стоит. На это занятие
и уйдёт.
</details>

<details><summary>Ограничения этого семинара, которые надо назвать вслух</summary>

Полный список того, где мы срезали угол, и в какую сторону это смещает выводы.

**Корпус в девять тысяч документов.** Меньше настоящего на пять порядков. Точный поиск здесь
занимает доли миллисекунды, то есть задача, ради которой существует ANN, на нашем масштабе
не стоит вовсе. Все выводы про **соотношения** переносятся, про абсолютные миллисекунды — нет.

**Задержка померяна плохо, и это осознанно.** Батчем из двухсот запросов, на многопоточном
faiss, с прогревом только для точного поиска, без разброса и без интервалов. Это строго хуже
того, что мы требовали от себя на неделях 4 и 9. Смещение: батч завышает пропускную способность
относительно одиночного запроса, а отсутствие интервалов не даёт отличить разницу от шума —
и мы прямо видели строку, где приближённый индекс «обогнал» точный.

**Мы не измерили доуточнение.** Самый заметный пропуск: PQ показан в худшем для него виде.
Правильная конфигурация — сжатый отбор плюс точное доуточнение — не построена и не померяна.

**Только один срез компромисса.** Прогон по `nprobe` при фиксированном `nlist`, прогон
по `efSearch` при фиксированном `M`. Полная паретовская граница требует двумерного перебора,
и без неё сравнивать HNSW с IVF нельзя — мы и не сравниваем.

**Одна метрика конечного качества.** MRR@10 при одном релевантном документе на запрос.
Псевдозапросы те же, что на неделях 7 и 9, со всеми их смещениями.

**Память индекса не измерена, только векторов.** Граф HNSW и списки IVF занимают место сверх
самих векторов, и в таблице части 1 этого нет. Для HNSW при `M = 32` это ещё около
260 байт на вектор — нулевой слой держит `2M` рёбер, а не `M`, плюс хвост верхних слоёв —
то есть больше самих данных при сжатии.

<summary>Как сделать правильно, если есть бюджет</summary>
Добавить доуточнение к PQ, перебрать двумерные сетки, померять память индекса через
`faiss.write_index` и размер файла, и повторить замер задержки по одному запросу
с `faiss.omp_set_num_threads(1)`. Это часы счёта и полностью снимает четыре из шести пунктов.
</details>

## Шаг 0 · Пины и preflight

In [ ]:
# ПИНЫ — точнее, ОТКАЗ от них там, где они ломают Colab.
# Базовый стек образа (numpy, scipy, scikit-learn, matplotlib, torch) собран сам под себя.
# Понижать его нельзя: `pip install numpy==1.26.4` откатывает ОДИН numpy, а scipy и sklearn
# остаются собранными под numpy 2 — и первый же импорт падает с
# «ModuleNotFoundError: No module named 'numpy.char'». Ставим ТОЛЬКО то, чего в образе нет.
import importlib.util as _ilu, subprocess as _sp, sys as _sys

NEEDED = {"sentence_transformers": "sentence-transformers", "faiss": "faiss-cpu"}
_missing = [pkg for mod, pkg in NEEDED.items() if _ilu.find_spec(mod) is None]
if _missing:
    print("ставлю:", ", ".join(_missing))
    _sp.run([_sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    print("готово · если следующий импорт упадёт — Runtime → Restart session, потом эта ячейка снова")
else:
    print("всё нужное уже в образе Colab — ставить нечего")

import json, math, os, random, re, statistics, time
from pathlib import Path

import numpy as np
import faiss
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sentence_transformers import SentenceTransformer

In [ ]:
# ДАННЫЕ ЛЕКЦИИ. Занятие сверяет свои результаты с числами лекции, а те живут в папке
# data/ курса. В Colab её нет — и раньше ноутбук падал на первой же сверке с
# FileNotFoundError. Файлы крошечные, поэтому вшиты прямо сюда: ниже точная копия нужных
# data/*.json, сжатая zlib и записанная base64. Гейт _research/check_notebooks.py следит,
# чтобы копия совпадала с оригиналом байт в байт, — так что разойтись они не могут.
# Если рядом уже лежит настоящая папка курса (или задан DLS_DATA), берётся ОНА.
import base64 as _b64, os as _os, zlib as _zlib
from pathlib import Path as _Path

_DATA_DIR = _Path(_os.environ.get("DLS_DATA", "./data"))
_EMBEDDED = {
    "l9-hnsw.json": (
        "eNrdGltu48jx36co6EtCaJrdJPWYgYM4WU9mgLUHWBvJx8IY8NGWiKXYWpKyox0YSH5ygCAnyUcOkKPMSVJN6lFNkRSl"
        "iXcXAQy42aqud1VXdffnM4Dep1AGvTfQe39792eYpkKEK8iElwYzE+7lCi7Bg+F5IkMB2dyL4/NnmcYhQnqLGfRvLvkA"
        "ogT4+TcGzLwkPA/kfLHMPT8WcL0M4igUXgJhlOVeEogM+lkexpE/MOGPJa3HVM5BJHm6gsSCmVxkkEvIZwKCWGYiyyER"
        "0XTmy2UKyySPYuQnloEXwzxK3uCavmO6fDQafPn7PxLW56brTNzig/ctc2SN2MAAXiA24Pb2MuEokp8uc3H+KNNA4Bwg"
        "NKQCcca/Y/grMy2UfRZlgH+Kk5vrP7y/uv1wd2NAInNkIFsg78sF9IegNJMVGBIJ4gn5yrynKJm+LVZ+7MdyCrcDeEYl"
        "ITYvW80XucyjwIBsJp8TkAkKGs3989jLRRKs4KJYyKwvf/v3+ZMIcpnCT1LOB4olAeIRfkik/waZyNEE8DwTqZq9ZJDl"
        "SnleThUE/VIusExrAP4yV6A2/CCEAoUALRaFSBhitBDgF4gs8BbKUOuFqAwk/Z1AfJfKWD+JBN5dfbi7g8JjNmqz4ClD"
        "3HeF56BsizgKRAj+Cj6lovSni6lIPsUT/PZic7Eye0bhfxlaNhDKBauQCAT9XK4QXek08BvQcUB/x4oBF8ssvfCj5GKx"
        "ymcysQclBcSA2D/jED9izxdxht/fF984k1gFWDlmZMzJ2CZjh4zdXjF8KGd6gZRpmG2pKQjlHoQckBGAZdCP7fjBqIXm"
        "R0E7R0EzCm0fgraPgnYpNN9Bn9H/L2sVinCqKWyHiCiLnVUI1kLZzVBEXN4JqgUXMYvTCcpthrI74XJqcGl++ONSpCui"
        "REyRmyWW6WqgRepF0LXeekVqfCfLkNy5caICVAuJ5EP4F5zcytVTSV4hKpKuZtNyW6H4Fl4+06KChqEeiGUoVpVRYCg5"
        "qPeQFgP31GagsV5E6Z+iLMpFiD9srdAr0xtOYRrUREIU92qTI/Q/7xj28kJdmkheXrJraXPflEor9zAqcrnppXruoEQK"
        "sCgsCFFtrecVKVaZXVuo3CPJTy/GQQp2PQW7noJj2hbjlEJdjujN5ZO4l2sJziq87KuT1amT1amzlPBkdVr1wlpNwirb"
        "HadOXk+B11MgIfWrMRjvYDBeZzBeZ7BSwl+x/zv1FJwmChPmHknBrafg1lNgpjtmrLvBkmUcn1HITf5XVSTNzOKRkU+a"
        "Ai3Tqpocoe0GaLaD1hKnql61CulGy8TikdoNPz8u8kgmuhvU7ctnG1Ivm6KP76q+TZNz/yyxzl6JtKy534DvZaKcOZdJ"
        "vNo0QEVT4ltFib1XUvvOpjguugVUC3z56z9V+a+wgSpgIEhlhs1LUcpPvcXgrWpxRIp9gcK2XCyQhdnSB58b2AFg+V2w"
        "wIo53FqwrPaCGa73R4bCkRSIdswCblcBYsR1SZipHsKfICsKKE+XRU/T1xuagbl2r5oK2CcVsE8qYJ9s9z6pgH1SAfsu"
        "GQ/JeETGYzKeUFoaYfaL1dL2UbU0O6qWHh6Fe3gU35ODnBzJ+JCCOwfBx8dh11LZ6CD4iIKPD4JrO5t7oMsooiirL96K"
        "3yqVxVzMfbG3GfGDAu21M1W2obrn00LiYU+Kus12w691kF8tRDQv1cPBaHJPzYKas+jG0pxUI6qzwE7UWaUSo/vgg9G4"
        "qlIO8JNW2d1WVeos56RVbrdV/CRawxa/67xq3G3V6H+xatJt1aTiG9Zpy7r6FDtJjaM2JvejvrW532Xt4X5n/37pq8Jn"
        "u5X3VBFxve75dzt/a9tP9+xt+ekwhzd1xNv13zItjGnhu+4RfL0T2nUEzDa1KG/sCKr19LaiJiXIfn1uab+8NFmNdDyI"
        "7qymnK+TadQgE/9qkXijSEpfx8pEmoId/O6Y5FvroPVeT9Jho6SuaY9drstqdMI5fgWck2aL7GLkSJzMavFcbg/ZSUhZ"
        "I1LbZENuHx8Rk84RMWmK8kJHP1+c/z/p/0D0qkT/EXvagzFsNdlmZFrj8Vfapllo5pg2G42P9zrW2etYk2Ql6a8UrdlJ"
        "Ss2d4iTH5PduGO1mjO4puyDvrP1X3NmP9KpumnKakZ5ULzidNeU0aernLRh+CYeqT2Idbo8qpaUuKSnW9BujSkmj7x1k"
        "Q9Pviu5l7sVtF0bO4Qsjko5bbsG0pKL7eVVCZ59XdVy5EOFV1as2F14191qbQ93dyal4vHsWYrF/eCoeIVO/qONGD2yr"
        "fBSyfgiSekko52aGpupPrMnAgCjJxVSk59M0CqE8zFMvRb63DMd6+M+/jPK0NBTKvHBzyY31sxPIci9FtaqDUnWgWVB5"
        "d/Xd/fvru/vyZPZHuIRnjIIcAsQxMOH3+nuOy3KRa5YPI8pTXPoS4i3+8NtL3uUdRC5h9xLCKF54LL1cHeTOZSp2K9Xb"
        "DvU6Y5rK58yEO1QELKIkwX/IF/grZBAl86M4ylfbw1ilL9Qtamw9oRoue3MXS47Ge0qNxFe2ecGx6k9N9+/P6QkZb7mD"
        "pnDM7XbTPmlBR/ZZ3nZBTo4W7JZLeZt1Y49PuvHHSS9uu50u3VvRETjWIi7RSgs22+70UIHQtFqQUUnHLSLQVwhWJxHa"
        "nloQqqxFBgpmdyLKuzmd3SICPfFuc07WzajUXPaoBR3vZn3iwuNOKrE6hStn3Vhjw27h2ppNCLpRp8cq9hYzG9c+VeGH"
        "z6zc6oGVa45t/VFKtt7lam4A9EtIlVW3Wf66zP9F1qYPDMo94ipn9RelVeT8IHLWhJ0dxu68Kvbxq2KnEdmAnrpeE/qz"
        "jZ+9nL2c/RcZdx2n"
    ),
    "l9-ivf.json": (
        "eNrNVttuGzcQfddXDPQkoauVuBdZdqGiaWwXfrCS2EaBIjCM1S5tbb0ibXKVVAkE9KkfEPQL+yUdci+ibquVk6J9sYbe"
        "M5czJOfwcwOgeRfxsHkCzYtfziHkgZC08zwLWBp/ogK+g5AmCTwJPo7Zgw03fA5DOIYnHrNUWsCSWKZDF+4F/0QZglkq"
        "eBzhl+cZFXN4hphlIcIetGQaJfG4jWHEjILbGY0wmJMH08geZiTw20ymEISCSwnphMKYiwiLUQBiA1PV0CHRVVGpnDhL"
        "5vD3n1/A6br6V9AwSJIfXYzfs/v9/tH3hZsDQSJ56UugpTI4LOowGgiKiVW5bR3FzaMRu2fD6zeXlxc3N2en8PPZm8uz"
        "m6tfIZYQSElFSiOYUEEhYJGqEoN/oCxgIb1Ljk+A/h6EKRboABLWLC1NyxVRxsmCV6NTVUuc8V2vBlKOnWwp8ypgjz/N"
        "T7Hr78ntcBgSrPSPv4Amki4ZfuSzJALGU+xDEE7KZJheVZg1x0CFSTwd541CvpruFQ0S7F6+seevLq6vAY9I5+27wv+D"
        "zDNC64EyZHqH2RL7ad62m5Y+WZLPREjV4cJPEgmFk24GRRQ2ns8tyM8E7vtqEGiZOS3ozqTo4iHsPs3TCWduO8uBMTD+"
        "ZzRxkR0kXL/Xayh/ARzbt4yFk9u31gbStXsmcrAb6a0ie7uRxMyOiyqkEdOtitk3kaQK6ZvZK2M6Zsx+FdJdRZaMGga+"
        "iZcjfmDGfpQ+pUGs/Z9KwymNlTTl1Nm+88YeVfSoFsqI1d/KWE89owx3e8Ua9hqvM0Jzvk09SnGduzQfDTvFeTkabWsk"
        "2dUSc1Qc4KgvdHmj8B/EWOSBzT7rmKV9u+xP857PWLSOtIwF2eqWDRfVFz2O8g+LAtF0atSzP8v+4syFU13p8vItGsXf"
        "RTGdlvWWSptPTfmR0qcTcHrFkF0VVR/VeDqNUyUuhq4+Dn3UTxzoPqhToTQUhQIidBHxeKbQhnLexwI1xNUqIjvjeUfh"
        "lDBByy2kiGjDKwzSxqHMC8mHKcfo2j2TCVmMf9wf6PyAPwP1ozVj9dSBVm2ZO4/nJYtOyjvPUFRSKDOkwSM1q2bA79Vy"
        "asO5QAWNOZOaq9ZTaOk9hC74ud5UC4Bxwb2Kkb5EVQzp/pYxsIkaLFFHu1HHteqqN8SIUysYcWuxJMYdcGu1jNQbsKQi"
        "mlEaIfWYVmU1qValNblW5jWGxHHNcL1am1/xziC1chqxyEvVuDS83fpM6ujzhuHuNrzdRulODhB8r94lcA4dCMT5Budx"
        "2/mpfEGU98av/YLwjReEf8ALYnNnDnxTeBUxV0Jp6TMiGILOiheIceozlX+rPkS13h6ZDlyHAWPaw/u3pB9FcP2Fso2L"
        "U4+LmdirRWzwcmLmwt3HclCHpfsClvufauuUzZH+VZz3d3vbC6+qAd7XNmD/ydvoxtH/thv+N+3G/uO63hqn95+2plHM"
        "9kVj0fgHIn3a2A=="
    ),
    "l9-pq.json": (
        "eNq9WEtv20gSvudXFHyYkWcoSnxIlmIIWMf2BF5MEtnSzC5gGEaLbFuMSLaGTWajPIA9LWaugwHyA+a0173kntznR+SX"
        "THXz1RQpW/Iu1ge5H9XVVdVVX/fHt48A9q5d5uw9hr1xxNzEieE8IWHsvSGxx0INAhqwaAUBiec6TNkKWrNVe05CVwP6"
        "mjixBjx2fW+2/xgInIwGcOMzElsmvKJOzCL48q9fIRjZwJNZOsKB3YD7zcgE1wu4nKfEmctGSElEeSwkFiOz1weHhnHE"
        "PDeV80KXvoYbL+bYBANmq5jKCVs2eSfb8xWHwecP9gitELMOC5aolqM/YlyHiUN82k6W0IqomMFN0NtXFH046A/aLi4K"
        "RsM+/hv24YlQZ3UPTGhZ5ucP+4dgmLnMAH8HqUTPQIG+jQI6HJ0cwwj40vdiiOcUfkoohtALYwaBEggNcHNhXIJukBCC"
        "zx8WGBQek9ChEJOZTzG6SQA+Y4tkyTH+qGx83g5swIXE9/9igMfh4nR8cTo5fT49mp79eAotx4upKyLkD9szGjpzDUIW"
        "gzi0tksjdNRFIy8o8dHKm4i9oSF8d3Q2maDuQrEm7XdQ0WwF1yJIJHLmnVsaXvtD7BNfX670PU1mEGdJ5FCRROuSKASt"
        "mK2KNIFvoaoDWvneGnQSHnVmXthZruI5C639VD+uR91vsYmdE2wOtLQdYNvO2u4kJhH2zay/EO1eP+vJ/PguTU2csEx1"
        "fHyu6FGyReyEg++lEVwkDY5cSrHUGLGtJ4zAtNHyEdEf9otuxY4GSzCzqnPSGkVB1SDLlOPvtSYzMDMrZgy2tQKzt8GI"
        "wQYb+nZqA/5eydgQ1ykPSCbuZE6WZbQgjy4II2TrqghsmnEXlUostQU2trv6QXY60bpYHCU0myuzsKqsLJYbBAe7XVSg"
        "yHcsXYlj0MIOFpPPOF8dwl8///uWJRqcsOQNha8QMuaBh5g3HR89OwOzaxhYQs8o4UmEJfKK+Al9vJ7YnaIA9b3CWwzV"
        "31i0oG7pYg7AAjaw2ESd6hJKU9xIcSRFjxRIxbhHObQEjCIgoc6pCPrly6tL50pAz08JEXZ9b8KMxv+gWODZshW8RKhx"
        "ERNdOkNYwSUFyIIDLQIIYXYKPvsp4LizDMu/5jC9+OFUhXLcRYI2zJJYggzFnX1AUxH3vKgA9ALHEQcYCHc4AgITNhIu"
        "jeFwaWrQ1cDSwLhCzJjTYvWSRmJTviQOTd09yVFyBH/8fv2yGgGhDRsiEFZP7CR0lbgqbUPQPH7xfIL+HE/PXjzXBP4e"
        "jccXL/5+9uxIjIg7SMjJ5FC2sw5yDSLz8luudUuWOJmCfXl9Ao0iFmU3wpdffs6vzAz0MeJzSpb+SqBwkAY9B3v48s/f"
        "pL4lw6MX1ozPU7jdEQPzOXn6SkUWBW/kjW7esLRa1daF0yrOS69ISmWH/L+yHBVkrSutJtUtpazNUnYpZW6WMkqpbi5V"
        "sTgvgGaDy1bF+NJ8ZdOqdFeV7t4nbanS5n3ShiptldKbw7DRMuu+vcydLLN3ipGxIUbb+WE/PGbGTl5bO521vasfxsMz"
        "ZTfLzJ28Nup+VAonh9rGulH27WlN5d/bqvzVtcqw0d1qtbFJ03arrebVjcFwZz/KC0AJhlVD0W4NTo3aSNGw1vRPisu2"
        "MeDWVpipuGdvhZn2VvhrbURW3nTR1O8Xo7JQkKGQ47XqxPJ5VIvoHZrqMbaaI6o8HsRrOjvhvco1LyYOihdbyoCPWXjj"
        "3fL6qw2fjRlH9ql7SyMdsgc0vgeCTx9nyFbHNDrGoHQG0FqsUVqf3ZqtxT4IsY6IHL5v1Lc5KsGn/aeP9mGFxo5ymt0Z"
        "nx+mvPjpk2sSXxt0iJOZBZ8+iq7slIPw9Ak+HKnjBfhQe/oEHyeSaksmW1hxiIOGGLPLseL14RShyE/obZESdT5UY0R1"
        "NiJIRxkmlXhs4ER306gGzqRMVYMl1WaT77Xt3TGG5n/tT1XHLg4Z/bscQr0P8Kh2QMZmf+xGf+zBA93p23e5Yw/u9Ubh"
        "vevM98Gnc4czClve0ZfClTUOnL9GpxHxwjrGfO+zlYv0a9EOKAk5IAKwUHwKKmiRVjLHfv5Nx2yfVL42YZ1rcOO9lh+H"
        "kIMhiHgR0isz5RgoYSI1o5GkLlyHU/FZTgwggV6yKE6JFEFPb8MAUSwnQsc/XFycPp+qJE8MeyGNYo9A64/f3717/eXn"
        "X5137z79Z1/MiY9hbiKsxk4opYsvYTqcZQs5kkkqWNelObB1pIZmV+8PDnrY0Pv9/sGV4HYBQ9opghGysO2FDpJv7oW3"
        "BVjxu+9PYytmsh1/MbZiJkpeDTZLDbeSUnQNG+/ihUIGxZn/j2NQ2atMneabQcyLbzlKfaS5pIjX3q7VR/LGmftf+XtF"
        "dq5tp7bXFKmqKsrWV5l3rGo0JqsNcToite8AuCxqxk5R6/6/o6Z31yKg97aLXF8/6GlrI71dwpdCQhVWs4S88ULiF2Da"
        "HDjF7uamoW1VJxsCVeF7uoV/Wm3kPnY2qK0brK2rGaPERyKl+sk4n5xk6Kq+1CXM5p0MbPNuqUheW4/eP/oTNkPjvg=="
    ),
}

_DATA_DIR.mkdir(parents=True, exist_ok=True)
_new = [n for n, b in _EMBEDDED.items() if not (_DATA_DIR / n).exists()]
for _n in _new:
    (_DATA_DIR / _n).write_bytes(_zlib.decompress(_b64.b64decode(_EMBEDDED[_n])))
print(f"данные лекции: {len(_EMBEDDED)} файл(ов) в {_DATA_DIR} · "
      f"распаковано {len(_new)}, остальные уже лежали на месте")


In [ ]:
def preflight():
    problems = []
    for name in ("l9-hnsw", "l9-ivf", "l9-pq", "l9-metrics", "l9-bench"):
        if not Path(f"{DATA_DIR}/{name}.json").exists():
            problems.append(f"нет {DATA_DIR}/{name}.json -- сверка с лекцией невозможна.")
    # Наличие GPU в системе ничего не значит: индексы ниже создаются CPU-шными и на GPU
    # не переносятся. Проверять надо не железо, а то, что мы с ним делаем.
    threads = faiss.omp_get_max_threads()
    print(f"· faiss использует {threads} потоков -- задержка зависит от этого числа")
    if not EMB_PATH.exists():
        print("· эмбеддингов недели 7 нет: кодируем корпус здесь, это около полуминуты")
    for p in problems:
        print("!", p)
    print("preflight:", "ЧИСТО" if not problems else f"{len(problems)} замечани(я/й) -- читай выше")
    return not problems

## Шаг 1 · Конфигурация

⚠️ Ловушка E · **Задержка faiss зависит от числа потоков, а не только от индекса.**
Библиотека по умолчанию распараллеливает поиск по запросам через OpenMP. Наш замер батчем
из двухсот запросов на десяти потоках даёт совсем не ту цифру, что один запрос на одном
потоке, — а в проде запросы приходят по одному. Правильно мерить **и** пропускную способность
(запросов в секунду при полной загрузке), **и** задержку одиночного запроса: это разные числа,
и оптимизируют их по-разному. Мы меряем только первое и говорим об этом вслух.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

SMOKE = os.environ.get("SMOKE", "0") == "1"
N_DOCS = 2000 if SMOKE else 9000        # верхняя граница; реальный размер ограничен корпусом
N_QUERIES = 50 if SMOKE else 200
SIZES = [1500, 3000, 6000, N_DOCS]      # на чём смотрим масштабирование точного поиска
NPROBES = [1, 2, 4, 8, 16, 32]
PQ_MS = [8, 16, 48]
DATA_DIR = os.environ.get("DLS_DATA", "./data")
ARTIFACTS = Path(os.environ.get("ARTIFACTS", "./artifacts"))
EMB_PATH = ARTIFACTS / "corpus_emb.npy"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

RUN = {"seed": SEED, "smoke": SMOKE, "n_docs_max": N_DOCS, "n_queries": N_QUERIES,
       "faiss": faiss.__version__}
print(json.dumps(RUN, ensure_ascii=False))
preflight()

**Что видно.** В конфигурации напечатано число потоков faiss, и это не формальность: библиотека
многопоточна по умолчанию, и все миллисекунды ниже зависят от того, сколько ядер ей досталось.
Сравнивать надо не строки вывода, а **число потоков с тем, что будет у соседа**: на бесплатном
Colab обычно два ядра, на своей машине бывает восемь, и задержка отличается кратно. Механизм
прямой — поиск по ячейкам и по графу распараллеливается по запросам. Чего этот вывод
НЕ показывает: собран ли faiss с поддержкой AVX-512, а это ещё в разы. Что делать: везде ниже
смотреть на **отношения** задержек, а не на абсолютные значения, и не переносить их никуда.

## Шаг 2 · Корпус и запросы

Корпус сегодня больше, чем на неделях 7 и 9, и это сознательно: приближённый поиск
на полутора тысячах векторов не нужен, и мы это сейчас увидим.

In [ ]:
raw = fetch_20newsgroups(subset="all", remove=("headers", "footers", "quotes"),
                         random_state=SEED)
TOKEN = re.compile(r"[a-z]{2,}")
SENT = re.compile(r"(?<=[.!?])\s+")
pool = []
for t in raw.data:
    t = " ".join(t.split())
    if len(TOKEN.findall(t.lower())) < 80:
        continue
    ss = [s for s in SENT.split(t) if 12 <= len(s.split()) <= 25]
    if ss:
        pool.append((t, ss))
random.Random(SEED).shuffle(pool)

N = min(N_DOCS, len(pool))
DOCS = [p[0] for p in pool[:N]]
QUERIES = []
for i in range(N_QUERIES):
    text, ss = pool[i]
    s = max(ss, key=len)
    QUERIES.append(s.strip())
    DOCS[i] = text.replace(s, "").strip()
GOLD = list(range(N_QUERIES))

st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
t0 = time.perf_counter()
EMB = st.encode(DOCS, batch_size=128, normalize_embeddings=True,
                show_progress_bar=False).astype("float32")
QEMB = st.encode(QUERIES, normalize_embeddings=True,
                 show_progress_bar=False).astype("float32")
D = EMB.shape[1]

print(f"корпус: {N} документов (запрошено {N_DOCS}, доступно {len(pool)}) · "
      f"запросов: {N_QUERIES}")
print(f"кодирование: {time.perf_counter() - t0:.0f} c · размерность {D}")
print(f"матрица эмбеддингов: {EMB.nbytes / 1e6:.1f} МБ ({EMB.nbytes / N:.0f} байт на вектор)")
RUN["n"], RUN["dim"] = N, D

**Что видно.** Запрошено девять тысяч документов, доступно меньше — фильтр по длине отсекает
короткие сообщения, и корпус получился таким, каким получился. Сравнивать надо не запрошенное
с полученным, а **байты на вектор с размерностью**: 384 числа по четыре байта дают 1536 байт,
и это ровно то, что печатает последняя строка. Механизм тривиален, но именно из этого числа
растёт вся арифметика памяти в части 4. Чего этот вывод НЕ показывает: что корпус в девять
тысяч документов хоть сколько-то похож на прод — он меньше настоящего на пять порядков.
Что делать: запомнить 1536 байт на вектор. К концу занятия мы сожмём их до восьми.

<details><summary>Векторная база или библиотека — что выбирать и чем они различаются</summary>

Мы работаем с faiss — это **библиотека**, а не база данных. Разница существеннее, чем кажется,
и её стоит понимать до того, как выбирать инструмент.

**Что даёт библиотека.** Структуру данных в памяти процесса и функции поиска. Всё остальное —
твоё: персистентность, обновления, репликация, фильтрация по метаданным, авторизация,
мониторинг. faiss, hnswlib, ScaNN, Annoy — из этого ряда.

**Что даёт база.** Всё перечисленное плюс сеть, транзакционность в той или иной степени,
и — важнее всего — **фильтрацию по метаданным вместе с поиском**. Qdrant, Weaviate, Milvus,
pgvector.

**Почему фильтрация — главный водораздел.** Запрос «похожие документы, но только за последний
месяц и только на русском» нельзя выполнить, отфильтровав результат поиска: если из ста
кандидатов условию удовлетворяют два, ты получишь два ответа вместо ста. Правильная реализация
фильтрует **внутри** обхода графа или при выборе ячеек, и это нетривиально: жёсткий фильтр
разрывает связность графа HNSW, и обход перестаёт находить путь. Библиотеки этого не умеют,
базы — умеют с оговорками.

**pgvector отдельно.** Расширение PostgreSQL, дающее векторный индекс прямо в реляционной базе.
Проигрывает специализированным по скорости на больших корпусах и выигрывает всем остальным:
у тебя уже есть транзакции, бэкапы, права доступа и SQL для фильтров. Для корпусов до нескольких
миллионов это чаще всего правильный ответ, и его недооценивают.

**Практическое правило.** Если корпус помещается в память одного процесса, обновляется редко
и фильтров нет — библиотеки достаточно, и она проще. Как только появляется хотя бы одно
из трёх — обновления, фильтры, несколько машин — библиотека превращается в базу, которую
ты пишешь сам, и лучше взять готовую.
</details>

<details><summary>Три способа уместить индекс, когда он не влезает — и что выбирают на практике</summary>

Арифметика из части 1 не оставляет вариантов: миллиард векторов по 768 измерений во float32
не поместится никуда. Способов ровно три, и они комбинируются.

**Первый: шардирование.** Разложить корпус по машинам, запрос отправить всем, ответы слить.
Память решается линейно, зато появляется сеть в критическом пути и задача слияния выдач —
ровно та, которой мы занимались на неделе 9. Плюс проблема хвостовой задержки: ответ приходит
не раньше самого медленного шарда, и при десяти шардах девяносто девятый перцентиль каждого
становится обычным делом.

**Второй: сжатие.** PQ, скалярное квантование, бинаризация. Память падает в десятки раз,
качество — тоже, и часть 4 показывает, насколько. Почти всегда применяется вместе
с доуточнением.

**Третий: вынести на диск.** Держать в памяти только структуру навигации (граф или центроиды),
а сами векторы читать с SSD по мере надобности. Так устроен DiskANN. Работает, потому что
доуточнять надо сотню векторов, а не миллиард, и сто случайных чтений с NVMe — это доли
миллисекунды.

**Что выбирают на практике.** Комбинацию: IVF с PQ в памяти для отбора кандидатов плюс
доуточнение по несжатым векторам с диска. Шардирование добавляют, когда корпус перестаёт
помещаться даже в сжатом виде на одной машине. Порядок именно такой: сначала сжатие
и диск, потом шарды, потому что каждая новая машина добавляет не только память, но и сеть,
и хвостовые задержки, и оперирование.

**Число, которое стоит запомнить.** Доуточнение сотни кандидатов по несжатым векторам стоит
одного обращения к диску и возвращает почти всё качество, потерянное при сжатии. Это лучший
известный обмен в этой области, и не пользоваться им странно.
</details>

⚠️ Ловушка A · **Корпус вырос — метрики упадут.** На неделе 7 у нас было полторы тысячи
документов, сегодня почти девять. У правильного ответа стало в шесть раз больше конкурентов,
и MRR неизбежно снизится. Это **не** ухудшение системы: сравнивать метрики между корпусами
разного размера нельзя вовсе. Мы будем сравнивать только приближённые индексы с точным поиском
**на одном и том же** корпусе.

---

## Часть 1 · Когда точный поиск ещё выигрывает — 20 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 1.1 | Как растёт цена точного поиска? | меряем на четырёх размерах |
| 1.2 | Сколько памяти это стоит? | **замер без модели**: арифметика хранения |

### Шаг 1.1 · Число, с которым сравнивается всё остальное

Точный поиск — наш `BASE`. Он даёт **правильный** ответ по определению: приближённые индексы
будут сравниваться именно с ним, а не с истиной.

In [ ]:
def exact_search(emb, q, k=100):
    t0 = time.perf_counter()
    sims = q @ emb.T
    top = np.argpartition(-sims, min(k, emb.shape[0] - 1), axis=1)[:, :k]
    order = np.argsort(-np.take_along_axis(sims, top, 1), axis=1)
    idx = np.take_along_axis(top, order, 1)
    return idx, (time.perf_counter() - t0) / len(q) * 1000

def recall_at(idx, gold, k):
    return float(np.mean([1.0 if g in idx[i, :k] else 0.0 for i, g in enumerate(gold)]))

def mrr_at(idx, gold, k=10):
    out = []
    for i, g in enumerate(gold):
        pos = np.where(idx[i, :k] == g)[0]
        out.append(1 / (pos[0] + 1) if len(pos) else 0.0)
    return float(np.mean(out))

exact_search(EMB[:500], QEMB)      # прогрев: первый вызов включает инициализацию BLAS
print(f"{'N':>7} {'мс/запрос':>11} {'МБ':>7} {'Recall@100':>12} {'MRR@10':>9}")
scaling = {}
for n in SIZES:
    n = min(n, N)
    sub = EMB[:n]
    idx, ms = exact_search(sub, QEMB)
    scaling[n] = {"ms": ms, "mb": sub.nbytes / 1e6,
                  "recall": recall_at(idx, GOLD, 100), "mrr": mrr_at(idx, GOLD)}
    print(f"{n:>7} {ms:>11.3f} {sub.nbytes / 1e6:>7.1f} "
          f"{scaling[n]['recall']:>12.3f} {scaling[n]['mrr']:>9.4f}")

EXACT, MS_EXACT = exact_search(EMB, QEMB)
BASE = mrr_at(EXACT, GOLD)
print(f"\nBASE -- точный поиск на всём корпусе: MRR@10 = {BASE:.4f} при {MS_EXACT:.3f} мс/запрос")
RUN["scaling"] = {str(k): v for k, v in scaling.items()}
RUN["base_mrr"], RUN["exact_ms"] = BASE, MS_EXACT

**Что видно.** Время растёт с размером корпуса, а метрики **падают** — и вот это второе важнее.
Сравнивать надо не строки таблицы по времени, а **колонку MRR сверху вниз**: при росте корпуса
в шесть раз качество проседает примерно на треть, потому что у правильного ответа стало
в шесть раз больше конкурентов. Механизм не имеет отношения к скорости и целиком объясняет,
почему сравнивать метрики между корпусами разного размера бессмысленно. Чего эта таблица
НЕ показывает: главного для сегодняшнего дня — время везде исчисляется **долями миллисекунды**.
На таком корпусе точный поиск — вообще не проблема. Что делать: запомнить `BASE`
и понимать, что дальше мы будем экономить доли миллисекунды. Смысл в том, что на корпусе
в миллион раз большем эта же арифметика даёт секунды.

<details><summary>Почему приближённый поиск не нужен на маленьком корпусе — и где граница</summary>

Наши абсолютные числа выглядят разочаровывающе: точный поиск и так занимает десятые доли
миллисекунды. Это не недостаток замера, а важный факт: **на девяти тысячах векторов
приближённый поиск не нужен**.

**Где граница.** Точный поиск на `N` векторах размерности `D` — это `N × D` умножений
на запрос, и современный BLAS выжимает из этого десятки гигафлопс. При `D = 384` и миллионе
векторов получается порядка сотни миллионов операций — единицы миллисекунд. То есть **до
миллиона векторов точный поиск вполне жизнеспособен**, особенно если запросов немного.

**Что меняется дальше.** С ростом `N` линейно растут и время, и память. Время можно купить
железом, память — нет: она упирается в физический объём машины. Поэтому настоящая граница
проходит не по времени, а по памяти, и наступает раньше.

**Практическое следствие, которое экономит недели.** Прежде чем строить ANN-индекс, посчитай:
сколько у тебя векторов, сколько запросов в секунду, сколько памяти. Если корпус меньше
миллиона, а нагрузка — десятки запросов в секунду, точный поиск проще, точнее и не требует
подбора `nprobe`, `efSearch` и кодовых книг. Половина внедрений векторных баз в небольших
проектах — это решение задачи, которой нет.

**Почему мы всё-таки строим индексы сегодня.** Потому что понимать, чем платят, надо **до**
того, как корпус вырастет. Все наблюдения этого занятия про соотношения — согласие против
конечной метрики, память против качества — переносятся на любой масштаб. Не переносятся только
миллисекунды.
</details>

### Шаг 1.2 · Замер без модели: арифметика памяти

**Замер без модели.** Сколько памяти нужно, чтобы просто **хранить** векторы? Ответ не зависит
ни от какого индекса, ни от какой модели поиска — это чистая арифметика, и именно она
определяет, какие решения вообще возможны.

In [ ]:
def index_gb(n_vectors, dim, bytes_per_dim=4):
    return n_vectors * dim * bytes_per_dim / 1e9

print(f"{'корпус':>14} {'dim=384':>10} {'dim=768':>10} {'dim=1536':>10}")
for n, label in ((1e6, "1 млн"), (1e7, "10 млн"), (1e8, "100 млн"), (1e9, "1 млрд")):
    print(f"{label:>14} {index_gb(n, 384):>9.1f}Г {index_gb(n, 768):>9.1f}Г "
          f"{index_gb(n, 1536):>9.1f}Г")
print()
RAM = 64
for dim in (384, 768, 1536):
    fits = RAM * 1e9 / (dim * 4)
    print(f"в {RAM} ГБ памяти помещается при dim={dim:>4}: {fits / 1e6:>6.1f} млн векторов "
          f"(float32, без индекса)")
print("\nи это ТОЛЬКО сами векторы: HNSW добавляет граф, IVF -- списки, всё это сверху")
RUN["mem_1e9_768gb"] = index_gb(1e9, 768)

**Что видно.** Миллиард векторов по 768 измерений — это три терабайта, и ни в какую разумную
машину они не помещаются. Сравнивать надо не строки таблицы между собой, а **любую из них
с объёмом памяти, который у тебя есть**: в шестьдесят четыре гигабайта влезает порядка двадцати
миллионов векторов размерности 768, и это верхняя граница, потому что сам индекс ещё не учтён.
Механизм чисто арифметический и потому непреодолимый: `N × D × 4` байта, и уменьшать можно
только множители. Чего этот расчёт НЕ показывает: что делать, когда не влезает — три ответа
(шардировать, сжать, вынести на диск), и сжатие мы разберём в части 4. Что делать: считать
эту таблицу **первой**, до выбора индекса. Она сразу отсекает половину вариантов.

<details><summary>Бюджет задержки в проде: куда на самом деле уходят миллисекунды</summary>

Мы меряем поиск изолированно. В работающей системе он — один из десятка шагов, и его доля
меньше, чем принято думать. В `data/l9-latency.json` лежит представительная раскладка;
разобрать её полезно, потому что она меняет приоритеты оптимизации.

**Типичный состав запроса.** Приём и парсинг запроса, его нормализация и, возможно,
переписывание моделью. Кодирование запроса энкодером. Поиск по индексу. Доуточнение.
Переранжирование кросс-энкодером. Сборка сниппетов — а это чтение самих документов, часто
из другого хранилища. Рендер ответа. Сеть между всеми участниками.

**Где на самом деле время.** Поиск по индексу — единицы миллисекунд. Кодирование запроса —
столько же. Переранжирование — десятки. Сниппеты — десятки, потому что это случайные чтения.
Сеть между сервисами — единицы на каждый переход, и переходов много. Итог: доля ANN-поиска
в общем бюджете часто меньше десяти процентов.

**Что отсюда следует.** Ускорение поиска вдвое даёт ускорение продукта на проценты — ровно
та же арифметика, что в ловушке B на неделе 3. Оптимизировать надо то, что доминирует,
а доминирует обычно переранжирование и сборка ответа.

**Когда ANN всё-таки критичен.** Когда он не помещается в память: тогда речь не о миллисекундах,
а о том, работает система вообще или нет. Именно поэтому в части 1 мы считали память, а не
время, — и именно поэтому вывод занятия сформулирован через память.

**Хвост важнее среднего.** Средняя задержка почти ничего не значит для пользователя;
значит девяносто девятый перцентиль. У ANN-индексов хвост бывает тяжёлым: запрос, попавший
на границу ячеек, просматривает больше кандидатов. Меряй перцентили, а не среднее, — мы
сегодня померяли среднее и это тоже названо в ограничениях.
</details>

⚠️ Ловушка F · **Опубликованные QPS всегда относятся к конкретной конфигурации памяти.**
Работа, показывающая миллион запросов в секунду на миллиарде векторов, почти наверняка
использует сжатие и распределение по десяткам машин. Цитировать её число рядом со своим
однопроцессным замером бессмысленно — это разные системы, а не разные реализации.

---

## Часть 2 · HNSW: граф малого мира — 25 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 2.1 | Как жадный обход находит соседа? | считаем на игрушке лекции по шагам |
| 2.2 | Что делает `efSearch`? | меряем согласие с точным поиском и задержку |

Идея HNSW: соединить векторы рёбрами так, чтобы от любого можно было добраться до любого
за логарифм шагов, и идти к запросу жадно, каждый раз переходя к ближайшему соседу.

In [ ]:
HN = json.load(open(f"{DATA_DIR}/l9-hnsw.json", encoding="utf-8"))["toy"]
nodes = np.array(HN["coords"]["nodes"], dtype=float)
q = np.array(HN["query"], dtype=float)
adj = {i: set() for i in range(len(nodes))}
for a, b in HN["edges"]:
    adj[a].add(b)
    adj[b].add(a)

def dist(i):
    return float(np.linalg.norm(nodes[i] - q))

def greedy(entry):
    cur, path = entry, [entry]
    while True:
        nxt = min(adj[cur], key=dist)
        if dist(nxt) >= dist(cur):
            return path
        cur = nxt
        path.append(cur)

path = greedy(HN["entry"])
brute = min(range(len(nodes)), key=dist)

assert path == HN["greedy"]["pathIdx"], f"путь разошёлся с лекцией: {path}"
assert brute == HN["bruteForce"]["nnIdx"], "перебор нашёл другой ответ"
assert abs(dist(brute) - HN["bruteForce"]["dist"]) < 1e-4, "расстояние разошлось"

print(f"запрос {HN['query']} · вход в граф: {HN['labels'][HN['entry']]}")
print(f"{'шаг':>5} {'узел':>6} {'расстояние':>12}  соседи (расстояние)")
for step, node in enumerate(path):
    nb = ", ".join(f"{HN['labels'][n]} {dist(n):.2f}" for n in sorted(adj[node]))
    print(f"{step:>5} {HN['labels'][node]:>6} {dist(node):>12.4f}  {nb}")
print(f"\nжадный обход: {HN['greedy']['hops']} перехода, посетил "
      f"{HN['greedy']['nodesVisited']} из {len(nodes)} узлов")
print(f"перебор:      ближайший {HN['labels'][brute]}, расстояние {dist(brute):.4f}")
print("сверка с data/l9-hnsw.json: путь и расстояние совпали")

**Что видно.** Жадный обход посетил три узла из шести и пришёл ровно туда же, куда перебор.
Сравнивать надо не путь с ответом, а **число посещённых узлов с размером графа**: экономия
вдвое на шести узлах кажется пустяком, но она растёт как отношение логарифма к линейной
функции, и на миллионе узлов обход посещает сотни вместо миллиона. Механизм в структуре графа:
дальние рёбра позволяют быстро приблизиться, ближние — точно доуточнить. Чего эта игрушка
НЕ показывает: случая, когда жадность **промахивается**, — а она промахивается, и именно
поэтому в настоящем HNSW обход хранит очередь из `efSearch` кандидатов, а не одного текущего.
Что делать: смотреть на третью строку таблицы. Из `n2` был переход в `n2` же — обход
остановился, потому что все соседи дальше, и это единственный критерий остановки.

<details><summary>Как проверить, что индекс не деградировал — три дешёвые проверки</summary>

Индекс — единственный компонент системы, который может тихо испортиться без изменения кода.
Три проверки, которые стоит поставить в мониторинг с первого дня.

**Первая: согласие с точным поиском на выборке.** Раз в сутки взять сто случайных запросов
из лога, посчитать по ним точный поиск перебором (сто запросов — это секунды даже
на миллионном корпусе) и сравнить с тем, что вернул индекс. Падение согласия означает, что
что-то сместилось: распределение данных, кодовая книга, параметры. Это самая информативная проверка
и самая редко применяемая.

**Вторая: гистограмма размеров ячеек (для IVF) или степеней вершин (для HNSW).** Если
распределение поехало — например, появилась одна гигантская ячейка, — качество упало,
а согласие могло ещё не успеть измениться. Стоит одну строку после сборки.

**Третья: доля запросов с пустой или короткой выдачей.** При фильтрации по метаданным
в векторных базах это первый симптом того, что фильтр разрывает связность графа: индекс
возвращает меньше `k` результатов, потому что не может до них добраться. Молчаливая деградация
в чистом виде.

**Что произойдёт без этих проверок.** Корпус пополняется новыми документами из новой предметной
области. Кодовая книга PQ и центроиды IVF обучены на старом распределении и постепенно перестают ему
соответствовать. Качество падает на проценты в месяц, никаких ошибок не возникает, и обнаружится
это через полгода по жалобам, а не по мониторингу.

**Правило.** Индекс требует переобучения при смене распределения так же, как модель. Разница
в том, что у модели есть версия и её выкатывают, а индекс перестраивают молча — и потому
за ним надо следить отдельно.
</details>

<details><summary>Что в HNSW делает буква H: иерархия и почему без неё хуже</summary>

Мы разобрали жадный обход на одном слое. Настоящий HNSW строит **несколько** слоёв, и буква H
в названии — про это.

**Устройство.** Каждый вектор попадает на слой `l`, выбранный случайно с экспоненциально
убывающей вероятностью: почти все на нулевом, немногие на первом, единицы на верхних. Рёбра
строятся внутри каждого слоя. Поиск начинается на самом верхнем слое, где узлов мало и шаги
огромные, спускается вниз, каждый раз уточняя.

**Зачем.** Без иерархии жадный обход по одному плотному слою делает много мелких шагов —
сложность получается ближе к корню из `N`, чем к логарифму. Иерархия даёт «дальние перелёты»
наверху и «точную посадку» внизу; это прямая аналогия со скип-листом, и авторы её явно
называют.

**Что настраивают.** `M` — число рёбер на узел: больше рёбер, лучше связность, больше памяти
(граф весит `M × N` ссылок). `efConstruction` — сколько кандидатов рассматривается при вставке:
влияет на качество графа и на время сборки, но не на время поиска. `efSearch` — единственный
параметр, который меняется **после** сборки и потому единственный, которым можно крутить
компромисс в проде.

**Слабое место HNSW.** Удаление. Граф не поддерживает настоящего удаления: узел помечается
как удалённый, но остаётся в структуре и продолжает участвовать в навигации. При большой доле
удалений индекс деградирует и требует перестройки. Для корпусов с активным обновлением это
серьёзный аргумент в пользу IVF, где перестроить можно одну ячейку.

**И про память.** Граф — это не бесплатно, и считать его надо внимательно: параметр `M` задаёт
рёбра **верхних** слоёв, а нулевой слой, где живут все векторы, держит `2M` ссылок. При `M = 32`
и четырёх байтах на ссылку это `2·32·4 = 256` байт на вектор в нулевом слое плюс несколько байт
на редкие верхние — около 260 байт на вектор сверх самих данных. На корпусе из миллиарда —
порядка 260 гигабайт только на рёбра. Именно поэтому HNSW редко применяют без сжатия векторов.
</details>

⚠️ Ловушка C · **Жадный обход не гарантирует правильного ответа.** На нашей игрушке он совпал
с перебором, и это свойство конкретного графа, а не метода. Локальный минимум — обычное дело:
обход застревает в узле, все соседи которого дальше, хотя настоящий ближайший лежит в другой
части графа. `efSearch` управляет тем, сколько путей рассматривается параллельно, и потому
прямо покупает вероятность не застрять.

In [ ]:
def build_hnsw(emb, m=32, ef_construction=200):
    idx = faiss.IndexHNSWFlat(emb.shape[1], m, faiss.METRIC_INNER_PRODUCT)
    idx.hnsw.efConstruction = ef_construction
    t0 = time.perf_counter()
    idx.add(emb)
    return idx, time.perf_counter() - t0

def agreement_at(approx, exact, k=10):
    return float(np.mean([len(set(approx[i, :k]) & set(exact[i, :k])) / k
                          for i in range(len(approx))]))

hnsw, build_s = build_hnsw(EMB)
print(f"сборка HNSW (M=32): {build_s:.1f} c на {N} векторов "
      f"({build_s / N * 1e6:.0f} мкс/вектор)")
print()
print(f"{'efSearch':>9} {'мс/запрос':>11} {'согласие@10':>13} {'MRR@10':>9} {'потеря MRR':>12}")
hnsw_rows = {}
for ef in (16, 32, 64, 128, 256):
    hnsw.hnsw.efSearch = ef
    t0 = time.perf_counter()
    _, idx = hnsw.search(QEMB, 100)
    ms = (time.perf_counter() - t0) / N_QUERIES * 1000
    m_, a_ = mrr_at(idx, GOLD), agreement_at(idx, EXACT)
    hnsw_rows[ef] = {"ms": ms, "agree": a_, "mrr": m_}
    print(f"{ef:>9} {ms:>11.3f} {a_:>13.3f} {m_:>9.4f} {(m_ / BASE - 1) * 100:>11.1f}%")
print(f"{'точный':>9} {MS_EXACT:>11.3f} {1.0:>13.3f} {BASE:>9.4f} {0.0:>11.1f}%")
RUN["hnsw"] = {str(k): v for k, v in hnsw_rows.items()}
RUN["hnsw_build_s"] = build_s

<details><summary>L2, скалярное произведение и косинус: где они расходятся — числа из лекции</summary>

Ловушка выше говорит, что на ненормированных векторах это три разные метрики. В
`data/l9-metrics.json` лежит пример, где расхождение видно на четырёх числах, и его стоит
разобрать: он объясняет, почему выбор метрики — не техническая деталь.

**Пример расхождения.** Запрос `[3, 3]`. Кандидат `p1 = [1, 0]` лежит ближе по евклидову
расстоянию (3,61 против 4,24), кандидат `p2 = [6, 6]` идеально совпадает по направлению
(косинус 1,0 против 0,71). **Победители разные:** L2 выбирает `p1`, косинус выбирает `p2`.
Никакой ошибки здесь нет — метрики отвечают на разные вопросы.

**Что спрашивает каждая.** Евклидово расстояние: «насколько похожи векторы **как точки**»,
то есть с учётом длины. Косинус: «насколько совпадает **направление**», длина игнорируется.
Скалярное произведение: «направление, взвешенное длинами обоих», то есть длинный документ
получает преимущество автоматически.

**Почему для текстов обычно косинус.** Длина эмбеддинга у большинства энкодеров коррелирует
не с содержанием, а с длиной текста и его «типичностью». Учитывать её — значит ранжировать
по длине, чего никто не хотел. Отсюда нормировка и косинус как умолчание.

**Когда всё-таки скалярное произведение.** Когда модель обучена под него, и норма несёт смысл
специально: некоторые ретриверы кодируют «уверенность» именно в длину вектора. Модели MS MARCO
часто из этого ряда, и переключение их на косинус портит качество. Читай карточку модели.

**Ещё одна тонкость из тех же данных.** В третьем блоке файла — три кандидата, где топ-1
различается у всех трёх метрик: `d1` побеждает по L2, `d2` по косинусу, `d3` по скалярному
произведению. Три метрики, три разных ответа на одном наборе из трёх документов. Это самый
короткий из известных мне способов убедиться, что выбор метрики — часть постановки задачи,
а не деталь реализации.

**Что это значит для faiss.** `IndexFlatIP` считает скалярное произведение, `IndexFlatL2` —
евклид. Косинуса среди них нет вовсе: его получают, нормируя векторы и беря `IP`. Забыть
нормировать — значит молча получить третью метрику из трёх, и никакого предупреждения
не будет.
</details>

**Что видно.** С ростом `efSearch` согласие ползёт к единице, задержка растёт — но главное
в правой колонке. Сравнивать надо не согласие с единицей, а **согласие с потерей конечной
метрики**: они меняются непропорционально, и здесь согласие **пессимистично**. При малом
`efSearch` согласие заметно ниже единицы, а MRR просел на проценты — потому что промахи
случаются на дальних позициях, где они почти никому не мешают. Механизм: жадный обход теряет
в первую очередь соседей, которые и так были далеко. Отдельно посмотри на строку, где потеря
оказалась **положительной**: приближённый индекс дал MRR чуть выше точного. Это не улучшение,
а шум — напоминание, что разницы такого размера обсуждать нельзя без интервалов, которых
у нас сегодня нет. Чего эта таблица НЕ показывает: где именно промахи, на первой позиции или
на девяностой. Что делать: не выбирать `efSearch` по согласию — ни в оптимистичную сторону,
ни в пессимистичную. Выбирать по потере той метрики, ради которой существует система.

⚠️ Ловушка B · **«Recall ANN» — это не полнота.** В литературе по приближённому поиску полнота означает **долю совпадения с точным поиском**, а не долю найденных релевантных
документов. Мы называем это «согласие», чтобы не путать с Recall@k из недели 4. Система может
иметь ANN-полноту 0,99 и при этом находить мало релевантного — если сам точный поиск находил мало.

---

## Часть 3 · IVF: ячейки и `nprobe` — 25 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 3.1 | Как режут пространство? | считаем на игрушке лекции |
| 3.2 | Что даёт каждая следующая ячейка? | прогон по `nprobe` — самая наглядная кривая занятия |

IVF проще HNSW: кластеризуем векторы, запрос сравниваем с центроидами, смотрим внутрь
нескольких ближайших ячеек. Число просматриваемых ячеек — `nprobe` — единственная ручка.

In [ ]:
IV = json.load(open(f"{DATA_DIR}/l9-ivf.json", encoding="utf-8"))["toy"]
pts = np.array(IV["points"], dtype=float)
cents = np.array(IV["centroids"], dtype=float)
qq = np.array(IV["query"], dtype=float)

assign = [int(np.argmin(np.linalg.norm(cents - p, axis=1))) for p in pts]
cell_order = list(np.argsort(np.linalg.norm(cents - qq, axis=1)))
true_nn = list(np.argsort(np.linalg.norm(pts - qq, axis=1))[:IV["k"]])

assert assign == IV["assign"], f"назначение по ячейкам разошлось: {assign}"
assert cell_order == IV["cellRankByDist"], "порядок ячеек по близости разошёлся"
assert sorted(true_nn) == sorted(IV["trueNN"]), f"истинные соседи разошлись: {true_nn}"

print(f"запрос {IV['query']} · ячеек {IV['nlist']} · ищем {IV['k']} ближайших")
print(f"истинные ближайшие: {sorted(true_nn)} (лежат в ячейках "
      f"{sorted({assign[i] for i in true_nn})})")
print(f"\n{'nprobe':>7} {'ячейки':>10} {'найдено':>16} {'полнота':>9}")
for np_ in ("1", "2"):
    info = IV["probe"][np_]
    cells = cell_order[:int(np_)]
    found = sorted(i for i in true_nn if assign[i] in cells)
    assert found == info["found"], f"найденные разошлись при nprobe={np_}"
    print(f"{np_:>7} {str(cells):>10} {str(found):>16} {info['recall']:>9.4f}")
print("\nсверка с data/l9-ivf.json: назначения, порядок ячеек и обе полноты совпали")

**Что видно.** При одной просмотренной ячейке нашлись два соседа из трёх, при двух — все три.
Сравнивать надо не полноту с единицей, а **ячейки, в которых лежат истинные соседи, с теми,
которые мы просмотрели**: третий сосед физически находится в соседней ячейке, и никакое
качество поиска внутри первой его не достанет. Механизм в этом и состоит: IVF теряет ровно
тех соседей, кто оказался по другую сторону границы кластера, и таких тем больше, чем ближе
запрос к границе. Чего эта игрушка НЕ показывает: что делать с запросами, попадающими прямо
на границу, — а это худший случай, и именно ради него `nprobe` делают больше единицы даже
там, где кажется, что хватит одной. Что делать: заметить, что цена линейна по `nprobe`,
а выигрыш — нет.

<details><summary>Проклятие размерности: почему в 384 измерениях всё почти равноудалено</summary>

Наши игрушки двумерны, и в них геометрия ведёт себя интуитивно. В 384 измерениях она ведёт
себя иначе, и понимать это важно, потому что на этом стоит вся сложность приближённого поиска.

**Что происходит.** В пространстве высокой размерности расстояния между случайными точками
концентрируются: отношение расстояния до ближайшего соседа к расстоянию до дальнего стремится
к единице с ростом размерности. Проще говоря, **все точки становятся примерно одинаково
далеки друг от друга**, и понятие «ближайший» теряет контраст.

**Почему поиск всё-таки работает.** Потому что реальные эмбеддинги не заполняют пространство
равномерно. Они лежат на многообразии заметно меньшей размерности — тексты про космос
занимают одну область, про хоккей другую. Эффективная размерность данных много ниже
номинальной, и именно она определяет, насколько хорошо работают индексы.

**Как это проявляется на практике.** Индексы, отлично работающие на синтетических гауссовых
данных, могут проваливаться на реальных эмбеддингах, и наоборот. Поэтому ANN-Benchmarks
использует **настоящие** наборы (GloVe, SIFT, deep features), а не случайные векторы.

**Связь с IVF и HNSW.** Обоим методам нужна кластеризуемость данных: IVF — чтобы ячейки были
осмысленными, HNSW — чтобы жадный обход не блуждал. На данных, у которых кластерной структуры
нет, оба вырождаются в перебор с накладными расходами. Проверить наличие структуры дёшево:
посмотреть распределение расстояний до ближайшего и до случайного соседа. Если они близки,
ANN не поможет.

**Что стоит унести.** Приближённый поиск работает не вопреки размерности, а благодаря тому,
что данные её не используют. Это свойство **твоих** данных, и его надо проверить, а не
предполагать.
</details>

In [ ]:
NLIST = int(4 * math.sqrt(N))          # эвристика faiss: примерно 4*sqrt(N) ячеек
quantizer = faiss.IndexFlatIP(D)
ivf = faiss.IndexIVFFlat(quantizer, D, NLIST, faiss.METRIC_INNER_PRODUCT)
t0 = time.perf_counter()
ivf.train(EMB)
ivf.add(EMB)
build_ivf = time.perf_counter() - t0
print(f"сборка IVF: {build_ivf:.1f} c · nlist={NLIST} · "
      f"в среднем {N / NLIST:.0f} векторов на ячейку")
print()
print(f"{'nprobe':>7} {'доля корпуса':>14} {'мс/запрос':>11} {'согласие@10':>13} "
      f"{'MRR@10':>9} {'потеря':>9}")
ivf_rows = {}
for np_ in NPROBES:
    ivf.nprobe = np_
    t0 = time.perf_counter()
    _, idx = ivf.search(QEMB, 100)
    ms = (time.perf_counter() - t0) / N_QUERIES * 1000
    m_, a_ = mrr_at(idx, GOLD), agreement_at(idx, EXACT)
    ivf_rows[np_] = {"ms": ms, "agree": a_, "mrr": m_, "share": np_ / NLIST}
    print(f"{np_:>7} {np_ / NLIST:>13.1%} {ms:>11.3f} {a_:>13.3f} {m_:>9.4f} "
          f"{(m_ / BASE - 1) * 100:>8.1f}%")
print(f"{'точный':>7} {1.0:>13.1%} {MS_EXACT:>11.3f} {1.0:>13.3f} {BASE:>9.4f} {0.0:>8.1f}%")
RUN["ivf"] = {str(k): v for k, v in ivf_rows.items()}
RUN["nlist"] = NLIST

**Что видно.** Это самая наглядная таблица занятия. Сравнивать надо **вторую колонку
с последней**: просматривая ничтожную долю корпуса, мы теряем часть качества, и обе величины
растут вместе, но с разной скоростью. При `nprobe = 1` потеря огромна — запрос видит только
свою ячейку, и правильный ответ часто оказывается за границей. Дальше каждая следующая ячейка
добавляет всё меньше, и кривая выполаживается. Механизм тот же, что на игрушке: теряются
соседи по ту сторону границы кластера, и с ростом `nprobe` границ становится всё меньше.
Чего эта таблица НЕ показывает: разброса — все числа посчитаны на одном наборе запросов,
и доверительных интервалов у них нет. Что делать: выбрать рабочую точку по **потере конечной
метрики**, а не по согласию, и убедиться, что она на пологой части кривой, а не на обрыве.

<details><summary>Как правильно мерить кривую «полнота против скорости»</summary>

Таблица `nprobe` — это одна точка зрения на компромисс. Стандартный способ представления
в литературе другой, и понимать его полезно, потому что все сравнения библиотек делаются так.

**Кривая `recall-QPS`.** По горизонтали — согласие с точным поиском (в литературе `recall`),
по вертикали — запросов в секунду, обычно в логарифмической шкале. Каждая точка — конфигурация
индекса. Метод A лучше метода B, если его кривая лежит выше и правее **на всём диапазоне**,
а не в одной точке. Именно так устроены ANN-Benchmarks.

**Почему одна точка ничего не значит.** Утверждение «наш индекс даёт полноту 0,95 при
10 000 QPS» бессмысленно без кривой: возможно, у конкурента при той же полноте двадцать тысяч,
а возможно, он вообще не умеет забираться выше 0,9. Сравнивать надо кривые, и потому в честных
работах их и публикуют.

**Что мы сделали не так.** Мы построили таблицу по одному параметру (`nprobe`) при
фиксированных остальных. Полная кривая требует перебора и `nlist`, и `nprobe`, и, для HNSW,
`M` вместе с `efSearch`. Наша таблица — срез, а не кривая, и на срезе методы сравнивать нельзя:
IVF с другим `nlist` мог бы выглядеть иначе.

**Чего в таких кривых обычно не хватает.** Времени сборки и памяти. Индекс, дающий отличную
кривую, но строящийся сутки и занимающий втрое больше, в проде проигрывает. ANN-Benchmarks
это отчасти учитывает, публикуя время сборки, — а вот память приходится считать самому.

<summary>Как сделать правильно, если есть бюджет</summary>
Перебрать сетку `nlist × nprobe` для IVF и `M × efSearch` для HNSW, для каждой конфигурации
записать (согласие, QPS, память, время сборки) и нарисовать паретовскую границу. Это десятки
прогонов по секунде каждый — то есть минуты на нашем корпусе.
</details>

<details><summary>Как выбирают nlist — и почему эвристика 4·√N именно такая</summary>

Число ячеек `nlist` — второй параметр IVF, и он выбирается при сборке, то есть менять его
потом дорого.

**Компромисс.** Много ячеек — каждая маленькая, просмотр одной дёшев, но вероятность, что
сосед за границей, выше, и нужен больший `nprobe`. Мало ячеек — каждая большая, просмотр дорог,
зато границ мало. Оптимум где-то посередине, и эвристика faiss `nlist ≈ 4·√N` даёт примерно
`√N/4` векторов на ячейку.

**Откуда корень.** При `nlist = √N` в каждой ячейке около `√N` векторов, и стоимость поиска
складывается из сравнения с `nlist` центроидами плюс просмотра `nprobe` ячеек по `N/nlist`
векторов. Минимизируя `nlist + nprobe·N/nlist` по `nlist`, получаем `nlist ≈ √(nprobe·N)`.
Множитель четыре — эмпирическая поправка на то, что кластеры неравномерны.

**Что ломает эвристику.** Сильно неравномерная плотность: если половина корпуса — почти
одинаковые документы, они соберутся в одну ячейку, и просмотр «одной ячейки» окажется
просмотром половины корпуса. Проверяется гистограммой размеров ячеек — одна строка кода после
сборки, и стоит делать всегда.

**Про обучение.** Центроиды обучаются k-means на выборке из корпуса. faiss требует
не менее `39 × nlist` точек и предупреждает, если их меньше, — то самое предупреждение,
о котором ловушка выше. При недостатке точек центроиды получаются случайными, ячейки —
неравномерными, и качество падает тихо.

**Практический вывод.** `nlist` выбирается по размеру корпуса и потом не трогается; `nprobe`
крутится в проде под нагрузку. Если хочется поменять `nlist`, это полная перестройка индекса,
то есть событие того же класса, что смена модели эмбеддингов.
</details>

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 3.4))
ax1.plot(NPROBES, [ivf_rows[p]["agree"] for p in NPROBES], "o--", color="#2E7D52",
         label="согласие с точным @10")
ax1.plot(NPROBES, [ivf_rows[p]["mrr"] / BASE for p in NPROBES], "o-", color="#3B6FD4",
         label="доля сохранённого MRR")
ax1.axhline(1.0, color="#111", lw=1, ls=":")
ax1.set_xscale("log", base=2); ax1.set_xlabel("nprobe (лог. шкала)")
ax1.set_ylabel("доля от точного поиска"); ax1.set_ylim(0, 1.05)
ax2 = ax1.twinx()
ax2.bar(NPROBES, [ivf_rows[p]["ms"] for p in NPROBES],
        width=[p * .4 for p in NPROBES], alpha=.18, color="#111")
ax2.set_ylabel("мс/запрос (столбики)")
ax1.legend(loc="lower right", fontsize=9)
plt.title("IVF: согласие и сохранённое качество растут по-разному")
plt.tight_layout(); plt.show()

**Что видно.** Две кривые идут рядом, но не совпадают, и синяя (доля сохранённого MRR) лежит
**выше** зелёной (согласие) почти везде. Сравнивать надо именно **вертикальный зазор между
ними** на каждом `nprobe`: он и есть та непропорциональность, о которой шла речь в части 2,
и знак у него на наших данных вполне определённый. Механизм: согласие считает попадания
в топ-10 как множества, а MRR взвешивает их обратным рангом; раз потери приходятся в основном
на хвост десятки, согласие падает, а MRR почти нет. Чего график НЕ показывает: что так будет
всегда — стоит промахам сместиться в первые позиции, и кривые поменяются местами. Что делать:
рисовать обе кривые всегда и смотреть на **знак** зазора. По одному согласию рабочая точка
выбирается систематически смещённо, и в какую сторону — заранее неизвестно.

⚠️ Ловушка D · **faiss предупреждает и продолжает.** Если обучающих точек меньше, чем нужно
для выбранного числа кластеров, faiss печатает `WARNING clustering ... please provide at least ...`
и **всё равно строит индекс**. Качество при этом тихо хуже: центроиды обучены на недостаточных
данных. Предупреждение легко пропустить среди прогресс-баров, а падения не будет. Правило:
после каждой сборки индекса смотреть в вывод, а не только на то, что код не упал.

⚠️ Ловушка D · **`METRIC_INNER_PRODUCT` против `METRIC_L2`.** Мы нормировали векторы,
и потому скалярное произведение эквивалентно косинусу, а евклидово расстояние даёт **тот же
порядок**. На ненормированных векторах это три разные метрики и три разных ответа. faiss
не проверяет, нормированы ли твои векторы, — он посчитает то, что попросили.

---

## Часть 4 · PQ: память против качества — 25 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 4.1 | Как устроено квантование произведения? | считаем арифметику сжатия по лекции |
| 4.2 | Что происходит с качеством? | меряем три степени сжатия |

PQ режет вектор на `m` кусков, каждый заменяет номером ближайшего центроида из обученной
кодовой книги. Вместо `D` чисел с плавающей точкой хранится `m` байт.

In [ ]:
PQD = json.load(open(f"{DATA_DIR}/l9-pq.json", encoding="utf-8"))
toy = PQD["toy"]

def pq_bytes(m, k=256):
    return m * int(math.log2(k)) // 8

assert toy["bytesFloat32"] == toy["D"] * 4, "арифметика float32 разошлась"
assert pq_bytes(toy["m"], toy["k"]) == toy["bytesPQ"], "байты PQ разошлись с лекцией"
assert toy["bytesFloat32"] // toy["bytesPQ"] == toy["compression"], "сжатие разошлось"

print(f"игрушка лекции: D={toy['D']}, m={toy['m']} кусков по {toy['dStar']} измерения, "
      f"кодбук из {toy['k']} центроидов")
print(f"  было {toy['bytesFloat32']} байт -> стало {toy['bytesPQ']} байт · "
      f"сжатие {toy['compression']}x")
print()
print(f"{'конфигурация':>28} {'байт/вектор':>13} {'сжатие':>9} {'индекс на 1 млрд':>18}")
for cfg in PQD["memoryConfigs"]["configs"]:
    print(f"{'dim=' + str(cfg['dim']) + ', m=' + str(cfg['m']) + ', k=' + str(cfg['k']):>28} "
          f"{cfg['bytesPQ']:>13} {cfg['compression']:>8}x {cfg['indexGB_at_1e9']:>17} ГБ")
print("\nсверка с data/l9-pq.json: арифметика игрушки совпала")

**Что видно.** Сжатие в тридцать два раза превращает три терабайта из части 1 в девяносто шесть
гигабайт — то есть в нечто, помещающееся в одну большую машину. Сравнивать надо не байты
с байтами, а **последнюю колонку с объёмом памяти сервера**: именно этот переход и делает
поиск по миллиарду векторов возможным на разумном железе. Механизм: вместо `D` чисел хранится
`m` номеров центроидов, и каждый номер занимает столько бит, сколько нужно для нумерации
кодовой книги. Чего эта таблица НЕ показывает: цены. Сжатие в тридцать два раза не бывает бесплатным,
и следующая ячейка показывает, сколько именно оно стоит. Что делать: помнить, что PQ хранит
**приближение** вектора, а не сам вектор, — расстояния после этого считаются с ошибкой.

<details><summary>Как выбрать индекс за пять минут — дерево решений</summary>

Собранное из всего занятия, в виде, которым можно пользоваться.

**Шаг 1. Посчитай память.** `N × D × 4` байт. Влезает в память одной машины с двукратным
запасом? Переходи к шагу 2. Не влезает — сразу к шагу 4.

**Шаг 2. Сколько векторов?** Меньше миллиона и нагрузка умеренная — бери **точный поиск**.
Он проще, точнее, не требует подбора и не деградирует. Половина внедрений векторных баз
в небольших проектах решает несуществующую задачу.

**Шаг 3. Больше миллиона, но влезает.** Бери **HNSW**, если корпус обновляется редко: лучшая
кривая `recall-QPS`, единственный параметр `efSearch` для настройки в проде. Бери **IVF**, если
обновляется часто: дешёвая вставка, перестроить можно одну ячейку.

**Шаг 4. Не влезает.** Сначала **скалярное квантование** — четырёхкратная экономия почти даром.
Всё ещё не влезает — **IVF-PQ с доуточнением** по несжатым векторам с диска. Всё ещё нет —
шардирование, и вместе с ним слияние выдач и хвостовые задержки.

**Шаг 5. Нужны фильтры по метаданным?** Тогда библиотеки не хватит: фильтрация должна идти
внутри обхода, иначе из ста кандидатов после фильтра останется два. Смотри в сторону векторных
баз, а при корпусе до нескольких миллионов — в сторону pgvector, где фильтры уже есть в SQL.

**Шаг 6. Настрой рабочую точку.** Прогон по единственному параметру поиска, критерий — потеря
**конечной** метрики, а не согласие с точным поиском. Запиши выбранную конфигурацию рядом
с числами: без неё результат невоспроизводим.

**Чего в этом дереве нет.** Ни одного упоминания «какой индекс лучше». Лучшего нет — есть
подходящий под твои `N`, `D`, память, частоту обновления и наличие фильтров. Ответы на эти
пять вопросов известны до всякого эксперимента.
</details>

<details><summary>Обновление индекса: то, о чём не пишут в статьях</summary>

Все работы по ANN меряют качество и скорость **поиска** на статическом корпусе. В проде корпус
меняется каждый день, и половина инженерной боли — про это.

**Добавление.** У HNSW вставка дорогая: надо найти соседей нового узла и перестроить рёбра,
то есть выполнить поиск с `efConstruction` кандидатами. У IVF — дешёвая: посчитать ближайший
центроид и дописать в конец списка. У PQ — дешёвая, но с оговоркой: кодовая книга обучена на старых
данных, и новый вектор кодируется тем, что есть.

**Удаление.** Это худшее место. Ни HNSW, ни IVF не поддерживают настоящего удаления: помечают
как удалённый и продолжают хранить. Накопится процентов десять — и индекс тратит время
на мертвецов; накопится половина — деградация становится заметной. Лечение одно: периодическая
полная перестройка.

**Дрейф распределения.** Центроиды IVF и кодовая книга PQ обучены на снимке корпуса. Добавили новую
предметную область — старые центроиды ей не соответствуют, документы валятся в одну-две ячейки,
и `nprobe`, подобранный раньше, перестаёт хватать. Обнаруживается по гистограмме размеров ячеек,
не обнаруживается никак иначе.

**Как живут на практике.** Сегментами, как Lucene: индекс состоит из нескольких неизменяемых
кусков, новое пишется в маленький свежий сегмент, поиск идёт по всем и сливает результаты
(снова слияние выдач, как на неделе 9), а фоновый процесс периодически объединяет сегменты
и перестраивает. Это же решает и удаление: мертвецы исчезают при слиянии.

**Что стоит унести.** Выбирая индекс, спрашивай не только «как быстро ищет», но и «как часто
придётся перестраивать и сколько это стоит». Для корпуса, обновляемого ежечасно, HNSW почти
всегда хуже IVF, притом что по кривой `recall-QPS` он выигрывает.
</details>

<details><summary>Альтернативы PQ: скалярное квантование, бинаризация, матрёшки</summary>

PQ — не единственный способ сжать векторы, и в последние годы у него появились конкуренты
попроще.

**Скалярное квантование (SQ).** Каждое измерение независимо приводится из float32 в int8:
находим минимум и максимум по измерению, растягиваем на 256 уровней. Сжатие ровно
четырёхкратное, ошибка мала, реализация — десять строк. Качество теряется на доли процента.
Это первое, что надо пробовать: четырёхкратная экономия почти бесплатно.

**Бинаризация.** Каждое измерение — один бит (знак после центрирования). Сжатие
тридцатидвухкратное, расстояние считается через XOR и popcount, то есть невероятно быстро.
Качество падает заметно, но при доуточнении по несжатым векторам схема работает и очень
популярна как первая ступень.

**Матрёшечные представления (Matryoshka).** Модель обучается так, что первые `k` координат
сами по себе — осмысленный эмбеддинг. Тогда «сжатие» — это просто взять префикс
вектора, без всякой кодовой книги и обучения. 768 измерений можно урезать до 128 с потерей
в единицы процентов, если модель обучена матрёшечно. Требует поддержки со стороны модели,
и она есть у нескольких современных энкодеров.

**Как выбирать.** По порядку возрастания сложности: сначала матрёшка (если модель умеет),
потом SQ, потом PQ, потом бинаризация с доуточнением. PQ выигрывает на очень больших корпусах,
где нужно сжатие сильнее четырёхкратного, и цена обучения кодовой книги амортизируется.

**Общее для всех.** Любое сжатие обязано измеряться **конечной** метрикой, а не ошибкой
восстановления вектора. Малая ошибка восстановления не гарантирует сохранения порядка соседей,
и наоборот — ровно та же мысль, что с согласием против MRR в частях 2 и 3.
</details>

<details><summary>ADC: как считают расстояния, не распаковывая коды</summary>

Главная хитрость PQ не в сжатии, а в том, что расстояния считаются **прямо по кодам**,
без восстановления векторов. Без этого сжатие было бы бесполезно: распаковка миллиона векторов
съела бы всю экономию.

**Как это работает.** Запрос тоже режется на `m` кусков. Для каждого куска заранее считается
расстояние до **всех** `k` центроидов соответствующей кодовой книги — получается таблица
`m × k` чисел, при `m = 96` и `k = 256` это около двадцати пяти тысяч значений. Считается она
один раз на запрос.

**Дальше — только сложение.** Расстояние от запроса до документа равно сумме `m` чисел,
выбранных из таблицы по кодам документа. Никаких умножений, только `m` обращений к памяти
и `m` сложений. Это на порядок дешевле честного скалярного произведения по `D` измерениям.

**Отсюда название.** Asymmetric Distance Computation: запрос остаётся **несжатым**, документ
сжат. Асимметрия принципиальна — если сжать и запрос, ошибка удвоится, а выигрыша не будет,
потому что таблица считается один раз и её стоимость амортизируется.

**Что это даёт по числам.** При `D = 768` честное произведение — 768 умножений и сложений.
ADC при `m = 96` — 96 сложений плюс построение таблицы. То есть примерно восьмикратная экономия
операций **сверх** тридцатидвухкратной экономии памяти. Именно поэтому PQ быстр, а не только
компактен.

**Чего он не чинит.** Ошибка приближения остаётся: расстояние считается до **центроида**
куска, а не до самого вектора. Это и есть та потеря качества, которую мы измерили в таблице
выше, и никакая скорость её не компенсирует. Отсюда доуточнение.
</details>

In [ ]:
print(f"{'m':>5} {'байт/вектор':>13} {'сжатие':>9} {'обучение, с':>13} {'мс/запрос':>11} "
      f"{'MRR@10':>9} {'потеря':>9}")
pq_rows = {}
for m in PQ_MS:
    pq = faiss.IndexPQ(D, m, 8, faiss.METRIC_INNER_PRODUCT)
    t0 = time.perf_counter()
    pq.train(EMB)
    pq.add(EMB)
    train_s = time.perf_counter() - t0
    t0 = time.perf_counter()
    _, idx = pq.search(QEMB, 100)
    ms = (time.perf_counter() - t0) / N_QUERIES * 1000
    m_ = mrr_at(idx, GOLD)
    b = m  # 8 бит на код -> m байт
    pq_rows[m] = {"bytes": b, "ms": ms, "mrr": m_, "train_s": train_s}
    print(f"{m:>5} {b:>13} {D * 4 // b:>8}x {train_s:>13.1f} {ms:>11.3f} {m_:>9.4f} "
          f"{(m_ / BASE - 1) * 100:>8.1f}%")
print(f"{'нет':>5} {D * 4:>13} {1:>8}x {0.0:>13.1f} {MS_EXACT:>11.3f} {BASE:>9.4f} {0.0:>8.1f}%")
RUN["pq"] = {str(k): v for k, v in pq_rows.items()}

**Что видно.** Вот честная цена сжатия, и она высока. Сравнивать надо **колонку сжатия
с колонкой потери**: сильнейшее сжатие обваливает MRR более чем вдвое, умеренное отнимает
проценты. Механизм понятен из устройства: чем меньше кусков `m`, тем больше измерений
приходится на один код, тем грубее приближение вектора и тем сильнее искажаются расстояния.
Чего эта таблица НЕ показывает: главного трюка индустрии — PQ почти никогда не применяют
в одиночку. Его ставят первой ступенью, отбирают кандидатов по сжатым векторам, а потом
**доуточняют** порядок по несжатым, поднятым с диска для сотни выживших. Это возвращает
почти всё качество, платя одним обращением к диску. Что делать: не воспринимать эту таблицу
как «PQ портит поиск». Она показывает, во что обходится PQ **без** доуточнения, и потому
объясняет, зачем доуточнение существует.

<details><summary>Доуточнение: как вернуть почти всё, потерянное при сжатии</summary>

Таблица выше показывает PQ **без** доуточнения, и потому выглядит удручающе. В проде так
никто не делает, и разница принципиальна.

**Схема.** Индекс на сжатых векторах отбирает не десять кандидатов, а, скажем, пятьсот.
Для этих пятисот поднимаются **несжатые** векторы — из памяти, если влезают, или с диска —
и считаются точные расстояния. Итоговый порядок строится по точным числам.

**Почему это дёшево.** Пятьсот точных скалярных произведений по 768 измерений — это триста
восемьдесят тысяч операций, доли миллисекунды. Пятьсот случайных чтений с NVMe — около
миллисекунды. То есть доуточнение добавляет к запросу порядка миллисекунды и возвращает
качество почти до уровня точного поиска.

**Почему почти.** Потому что доуточнить можно только то, что отобрано. Если сжатый индекс
не вернул правильный документ в свои пятьсот, доуточнение его не создаст — это тот же потолок,
что на неделе 7 у переранжирования и на неделе 9 у слияния. Одна и та же мысль в третий раз:
**последняя ступень ограничена первой**.

**Как выбирают глубину доуточнения.** Ровно так же, как глубину переранжирования на неделе 7:
прогоном, глядя на конечную метрику и на цену. И ровно так же выясняется, что дальше некоторого
предела углублять бесполезно.

**Что это значит для нашей таблицы.** Строка `m=8` с потерей в шестьдесят процентов — это
не приговор PQ, а измерение того, насколько плохо восстанавливаются расстояния при сильном
сжатии. С доуточнением та же конфигурация потеряла бы единицы процентов, оставаясь
в сто девяносто два раза компактнее. Мы этого не измерили, и это самый заметный пропуск
занятия.
</details>

⚠️ Ловушка E · **Обучение кодовой книги — это тоже время и тоже данные.** Колонка «обучение»
не нулевая, и на реальном корпусе она измеряется минутами. Хуже: кодовая книга обучается на выборке
из корпуса, и если распределение сместится (добавили новую предметную область), кодовая книга
устареет, а качество упадёт тихо. Переобучение кодовой книги означает **полную переиндексацию**.

---

## Задания — 20 мин

**Про самопроверку честно:** пройденная самопроверка не гарантирует, что задание сделано
осмысленно, но проваленная гарантирует, что где-то ошибка.

### Задание 1 · Рабочая точка по правильному критерию

**Тезис.** *Согласие с точным поиском и потеря конечной метрики — разные величины, и то, какая
из них строже, надо проверять, а не считать очевидным.* Проверим на нашей же таблице IVF.

**Что сделать.** Найди `nprobe_agree` — наименьший `nprobe`, при котором **согласие@10**
не ниже 0,95, и `nprobe_mrr` — наименьший, при котором **потеря MRR** не превышает 5 %.
**Честно сравни** их и посчитай, во сколько раз отличается задержка.

**Что нужно получить.** `nprobe_agree`, `nprobe_mrr` (`int` из `NPROBES`), `ms_ratio` (`float`).

**Подсказка.** Всё нужное лежит в `ivf_rows[nprobe]`: ключи `agree`, `mrr`, `ms`. База — `BASE`.

**Прочитай до запуска.** Все исходы содержательны:
* `nprobe_mrr > nprobe_agree` — критерий по согласию выбрал более дешёвую и более плохую точку,
  то есть согласие оптимистично;
* совпали — на нашей сетке оба порога пересекаются в одной точке; это не значит, что величины
  совпадают, — сравни их **между** узлами сетки и увидишь разницу;
* `nprobe_mrr < nprobe_agree` — MRR восстанавливается раньше согласия, то есть согласие
  **пессимистично**: промахи приходятся на дальние позиции, где они почти ничего не стоят.

**Формулировка вывода.** Не «надо мерить MRR», а: **при каком свойстве промахов** согласие
оказывается оптимистичным, а при каком — пессимистичным.

In [ ]:
# --- твой код: ЗАДАНИЕ 1 ---
nprobe_agree = ...
nprobe_mrr = ...
ms_ratio = ...
# --- конец ---

assert nprobe_agree in NPROBES and nprobe_mrr in NPROBES, "оба значения -- из сетки NPROBES"
assert ivf_rows[nprobe_agree]["agree"] >= 0.95 - 1e-9, "согласие в выбранной точке ниже 0.95"
assert all(ivf_rows[p]["agree"] < 0.95 for p in NPROBES if p < nprobe_agree), \
    "нашёлся МЕНЬШИЙ nprobe с согласием >= 0.95 -- нужен наименьший"
assert ivf_rows[nprobe_mrr]["mrr"] >= 0.95 * BASE - 1e-9, "потеря MRR больше 5%"
assert all(ivf_rows[p]["mrr"] < 0.95 * BASE for p in NPROBES if p < nprobe_mrr), \
    "нашёлся МЕНЬШИЙ nprobe с потерей <= 5% -- нужен наименьший"
assert ms_ratio > 0, "отношение задержек положительно"
print(f"по согласию >= 0.95:      nprobe={nprobe_agree}, "
      f"{ivf_rows[nprobe_agree]['ms']:.3f} мс, потеря MRR "
      f"{(ivf_rows[nprobe_agree]['mrr'] / BASE - 1) * 100:+.1f}%")
print(f"по потере MRR <= 5%:      nprobe={nprobe_mrr}, "
      f"{ivf_rows[nprobe_mrr]['ms']:.3f} мс, согласие "
      f"{ivf_rows[nprobe_mrr]['agree']:.3f}")
print(f"задержка отличается в {ms_ratio:.2f} раза")
RUN["task1"] = {"nprobe_agree": nprobe_agree, "nprobe_mrr": nprobe_mrr, "ms_ratio": ms_ratio}

### Задание 2 · Что влезает в память

**Что сделать.** Посчитай, сколько векторов размерности 768 помещается в 64 ГБ при трёх
режимах хранения: float32, PQ с `m=96` и PQ с `m=48`. Затем определи `min_m` — наименьшее `m`
из `PQ_MS`, при котором **потеря MRR на нашем корпусе** не превышает 15 %.

**Что нужно получить.** `capacity` — словарь `{"float32": n, "pq96": n, "pq48": n}` с числом
векторов (`int`), `min_m` (`int` из `PQ_MS`).

**Подсказка.** PQ с `k=256` хранит один байт на кусок, то есть `m` байт на вектор.

**Прочитай до запуска.** Исходы:
* ёмкость растёт ровно во столько раз, во сколько сжатие, — так и должно быть, проверь себя;
* `min_m` оказался наибольшим из `PQ_MS` — сжатие дорого обходится даже на умеренных степенях;
* `min_m` — наименьший — на нашем корпусе PQ почти бесплатен, что было бы неожиданно и стоило
  бы перепроверить, не забыл ли ты обучить кодовую книгу.

**Формулировка вывода.** Не «PQ экономит память», а: **какой вопрос надо задать первым** —
про память или про качество — и почему порядок важен.

<details><summary>Шесть типов ловушек этого занятия — и почему тут доминирует тип E</summary>

Соберём сегодняшние ловушки и посмотрим, чем это занятие отличается от предыдущих.

**A · данных.** Корпус вырос вшестеро относительно недели 7, и метрики упали — сравнивать
их между размерами корпуса нельзя.

**B · метрики.** «Recall» в литературе по ANN означает согласие с точным поиском, а не полноту
относительно правды. Одно слово, два разных смысла, и путаница стоит дорого.

**C · интерпретации.** Жадный обход совпал с перебором на игрушке — это свойство конкретного
графа, а не гарантия метода.

**D · инструмента.** Две штуки, и обе про faiss: предупреждение о нехватке точек для обучения,
после которого библиотека **продолжает**; и метрика расстояния, которую faiss не проверяет
на согласованность с нормировкой векторов.

**E · замера.** Тоже две, и это главное отличие занятия: задержка зависит от числа потоков,
и батч из двухсот запросов меряет пропускную способность, а не задержку. Плюс прогрев,
без которого первая строка таблицы была вдвое хуже остальных, — мы это чинили прямо по ходу.

**F · переноса.** Опубликованные QPS всегда относятся к конкретной конфигурации памяти
и железа.

**Почему сегодня доминирует E.** Потому что впервые за курс мы меряем не качество, а **время
и память**. Качество измеряется на одних и тех же данных и потому воспроизводимо; время
измеряется на конкретном железе конкретной сборкой библиотеки в конкретный момент, и почти
всё, что можно сделать неправильно, относится к методике замера, а не к алгоритму.

Отсюда правило занятия: **всякий раз, когда результат — это время, половина работы уходит
на то, чтобы замер вообще что-то означал.** Прогрев, число потоков, размер батча, повторы,
разброс. Мы сделали из этого списка меньше половины и сказали об этом вслух.
</details>

In [ ]:
# --- твой код: ЗАДАНИЕ 2 ---
capacity = ...
min_m = ...
# --- конец ---

assert set(capacity) == {"float32", "pq96", "pq48"}, "нужны все три режима"
assert abs(capacity["pq96"] / capacity["float32"] - 32) < 0.01, \
    "PQ m=96 при dim=768 сжимает в 32 раза -- ёмкость обязана вырасти во столько же"
assert abs(capacity["pq48"] / capacity["pq96"] - 2) < 0.01, "m=48 вдвое компактнее m=96"
assert min_m in PQ_MS, "min_m -- из сетки PQ_MS"
assert pq_rows[min_m]["mrr"] >= 0.85 * BASE - 1e-9, "потеря больше 15%"
print(f"{'режим':>10} {'байт/вектор':>13} {'векторов в 64 ГБ':>20}")
for name, b in (("float32", 768 * 4), ("pq96", 96), ("pq48", 48)):
    print(f"{name:>10} {b:>13} {capacity[name]:>19,}")
print(f"\nнаименьшее m с потерей <= 15%: {min_m} "
      f"(потеря {(pq_rows[min_m]['mrr'] / BASE - 1) * 100:+.1f}%)")
RUN["task2"] = {"capacity": capacity, "min_m": min_m}

### Задание 3 · Словами: почему «полнота 0,95» ничего не обещает

**Что сделать.** Ответь **словами** на два вопроса:

1. Две системы приближённого поиска имеют одинаковое согласие с точным поиском — 0,95.
   Опиши ситуацию, в которой одна из них теряет 1 % конечной метрики, а вторая — 30 %.
   Что именно должно различаться в их промахах?
2. Мы измеряли согласие по топ-10. Что изменилось бы, меряй мы его по топ-100, и почему
   для выбора рабочей точки это важно?

**Прочитай до запуска.** `assert` проверяет объём, замену заглушки и упоминание позиции
промаха. Ответ без слова «позиция» или «ранг» проверку не пройдёт: без него первый пункт
не может быть верным.

**Формулировка вывода.** Не «согласие плохая метрика», а: при каком условии согласие
становится достаточным критерием и как это условие проверить.

In [ ]:
# --- твой код: ЗАДАНИЕ 3 ---
ANSWER = """
Впиши ответ сюда: минимум 80 слов, оба пункта, с упоминанием позиции промаха.
"""
# --- конец ---

assert len(ANSWER.split()) >= 80, "ответ короче 80 слов -- два пункта так не уместить"
assert "Впиши ответ" not in ANSWER, "заглушка не заменена"
assert "позици" in ANSWER.lower() or "ранг" in ANSWER.lower(), \
    "первый пункт невозможно объяснить, не сказав, ГДЕ находится промах"
print(f"ответ принят: {len(ANSWER.split())} слов")

---

## Итог занятия — 5 мин

* Посчитали, когда точный поиск перестаёт работать: не по времени, а по **памяти**. Миллиард
  векторов по 768 измерений — три терабайта, и это конец разговора.
* Собрали три индекса и сверили с доской жадный обход HNSW, ячейки IVF и арифметику PQ.
* Показали главное: **согласие с точным поиском и потеря конечной метрики — разные величины**.
  На наших данных согласие оказалось пессимистичным — промахи легли в хвост десятки, — но знак
  этого смещения заранее неизвестен, и потому выбирать рабочую точку надо по конечной метрике.
* Измерили цену сжатия. Она высока, и именно поэтому PQ в проде почти всегда идёт
  с доуточнением по несжатым векторам.

**Ограничение нашего замера, которое надо назвать вслух.** Корпус в девять тысяч документов
меньше настоящего на пять порядков, и на нём точный поиск занимает доли миллисекунды.
Все выводы про **порядок** величин переносятся, все выводы про абсолютные миллисекунды — нет.
Задержка померяна на многопоточном faiss без прогрева, на двухстах запросах и без
доверительных интервалов — то есть строго хуже того, что мы требовали от себя на неделях 4 и 9.
Это сознательное упрощение ради времени, и оно названо.

**Что мы будем и чего не будем замерять дальше.** На неделе 12 поверх выбранной сегодня
рабочей точки встанет RAG, и потеря полноты от приближённого поиска сложится с потерей
от чанкования. Складываются они не линейно, и разбирать это придётся заново.

In [ ]:
RUN["finished"] = True
(ARTIFACTS / "ann.json").write_text(json.dumps({
    "n": N, "dim": D, "nlist": NLIST,
    "exact_ms": MS_EXACT, "base_mrr": BASE,
    "hnsw": RUN["hnsw"], "ivf": RUN["ivf"], "pq": RUN["pq"],
    "working_point": {"index": "IVF", "nprobe": nprobe_mrr,
                      "ms": ivf_rows[nprobe_mrr]["ms"],
                      "mrr": ivf_rows[nprobe_mrr]["mrr"]},
    "run": RUN,
}, ensure_ascii=False), encoding="utf-8")
(ARTIFACTS / "run-ann.json").write_text(
    json.dumps(RUN, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"индексы -> {ARTIFACTS / 'ann.json'}")
print(f"замеры  -> {ARTIFACTS / 'run-ann.json'} ({len(RUN)} ключей)")
print(f"рабочая точка: IVF nprobe={nprobe_mrr}, {ivf_rows[nprobe_mrr]['ms']:.3f} мс, "
      f"MRR {ivf_rows[nprobe_mrr]['mrr']:.4f} "
      f"({(ivf_rows[nprobe_mrr]['mrr'] / BASE - 1) * 100:+.1f}% к точному)")
print("на неделе 12 RAG встанет поверх этой точки")

**Что видно.** В артефакт легла **рабочая точка** — не лучший индекс, а выбранная конфигурация
с её ценой. Сравнивать надо не индексы между собой, а **записанную точку с критерием,
по которому она выбрана**: она получена заданием 1, то есть по потере конечной метрики, а не
по согласию. Механизм важен для следующих занятий: любое сравнение на неделе 12 будет
проводиться при этой конфигурации, и без записанного `nprobe` оно окажется несопоставимым.
Чего этот вывод НЕ показывает: устойчивости точки — на другом корпусе или другой модели она
будет другой. Что делать: перевыбирать рабочую точку при каждой смене эмбеддингов. Кодовая книга
и центроиды обучены на **этих** векторах.

<details><summary>Пять семейств индексов одной таблицей — и что в ней главное</summary>

В `data/l9-compare.json` лежит сводная матрица из лекции: flat, HNSW, IVF, IVF-PQ, HNSW-PQ.
Стоит посмотреть на неё целиком, потому что в ней видно то, чего не видно в отдельных
измерениях.

**Flat.** Точный, без параметров, память равна данным, время линейно. Единственный, который
не требует ни обучения, ни сборки, ни настройки. Его недооценивают.

**HNSW.** Лучшая кривая полнота-скорость, память — данные плюс граф, вставка дорогая,
удаление отсутствует. Один параметр в проде: `efSearch`.

**IVF.** Хуже HNSW по кривой, зато дешёвая вставка, обновляемые ячейки и понятная механика.
Требует обучения центроидов и выбора `nlist` при сборке.

**IVF-PQ.** То же плюс сжатие. Единственный из пяти, который переживает миллиард векторов
на разумном железе. Требует обучения двух вещей — центроидов и кодовой книги — и потому сильнее
всех страдает от дрейфа распределения.

**HNSW-PQ.** Комбинация встречается реже: граф уже занимает память, и сжимать векторы,
оставляя граф, помогает меньше.

**Что в этой таблице главное.** Не строки, а **столбцы**: время сборки, память, качество,
обновляемость, число настраиваемых параметров. Работы по ANN обычно оптимизируют один столбец
(качество при фиксированной скорости) и молчат про остальные четыре. В проде решает как раз
сумма по всем пяти, и потому «лучший индекс по бенчмарку» и «правильный индекс для задачи» —
разные вещи заметно чаще, чем хотелось бы.
</details>

<details><summary>Что понадобится на неделе 12 и почему рабочая точка важнее индекса</summary>

В артефакт легли три семейства индексов и одна выбранная рабочая точка. Ценна именно последняя,
и вот почему.

**Что делает неделя 12.** Строит RAG: поиск отдаёт чанки, языковая модель на них отвечает.
Качество ответа зависит от того, попал ли нужный чанк в контекст, то есть от полноты поиска.
И вот здесь потеря от приближённого индекса складывается с потерей от чанкования.

**Почему складываются нелинейно.** Чанкование дробит документ на куски, и нужный ответ может
оказаться на границе двух чанков — тогда не поможет никакой поиск. Приближённый поиск теряет
часть кандидатов. Если оба механизма теряют **одни и те же** документы, общая потеря близка
к большей из двух; если разные — потери суммируются. Заранее неизвестно, и мерить придётся
вместе, а не по отдельности.

**Что было бы неправильно.** Взять на неделе 12 другую конфигурацию индекса (например, точный
поиск, потому что корпус маленький) и сравнить с сегодняшними числами. Тогда разница
в качестве RAG объяснялась бы сменой индекса, а не чанкованием, и вывод был бы про не то.
Рабочая точка записана, чтобы этого не случилось.

**Тонкость про эмбеддинги.** Кодовая книга PQ и центроиды IVF обучены на **этих** векторах. Сменится
модель эмбеддингов — индекс придётся строить заново, и рабочую точку выбирать заново. Записанная
конфигурация действительна ровно для пары «модель + корпус», и это стоит указывать рядом
с числами, что мы и делаем через `run.json`.

**Общий принцип занятия.** Индекс — не «ускоритель поиска», а компонент с собственным
качеством. Его конфигурация входит в описание системы наравне с моделью, и без неё результат
невоспроизводим.
</details>

---

## Решения

**Подглядеть — не поражение. Поражение — уйти с занятия, не поняв, где был затык.**

<details><summary>Задание 1 · рабочая точка</summary>

```python
nprobe_agree = next(p for p in NPROBES if ivf_rows[p]["agree"] >= 0.95)
nprobe_mrr = next(p for p in NPROBES if ivf_rows[p]["mrr"] >= 0.95 * BASE)
ms_ratio = ivf_rows[nprobe_mrr]["ms"] / ivf_rows[nprobe_agree]["ms"]
```

Согласие@10 считает пересечение множеств и не различает, потерян документ с первого места
или с десятого. MRR взвешивает обратным рангом, поэтому потеря первого места стоит в десять
раз дороже потери десятого. Отсюда обе возможности:

**Согласие оптимистично**, когда промахи концентрируются в верхних позициях: множества
пересекаются на девяносто процентов, а MRR обваливается, потому что потерян именно первый
документ.

**Согласие пессимистично**, когда промахи уходят в хвост: индекс путает восьмое место
с двенадцатым, согласие@10 честно падает, а MRR не замечает почти ничего. На наших данных
происходит именно это — сравни в таблице IVF колонку потери с величиной `1 − согласие`:
потеря стабильно **меньше**.

Практическая проверка одна на оба случая: посчитать распределение позиций, на которых
приближённый индекс расходится с точным. Равномерно — согласие приемлемо; сконцентрированно
наверху — выбирать только по конечной метрике.
</details>

<details><summary>Задание 2 · память</summary>

```python
capacity = {
    "float32": int(64e9 // (768 * 4)),
    "pq96": int(64e9 // 96),
    "pq48": int(64e9 // 48),
}
min_m = next(m for m in sorted(PQ_MS) if pq_rows[m]["mrr"] >= 0.85 * BASE)
```

Порядок вопросов: **сначала качество, потом память.** Причина в том, что вопрос о памяти
имеет ответ всегда — сжимай сильнее, шардируй, вынеси на диск, — а вопрос о качестве может
не иметь ответа вовсе: если при допустимом уровне сжатия качество неприемлемо, задача
не решается этим способом, и надо менять подход, а не конфигурацию.

Начав с памяти, ты подберёшь конфигурацию под железо и обнаружишь потерю качества в конце,
когда всё уже построено. Начав с качества, ты сразу узнаешь верхнюю границу сжатия и будешь
выбирать железо под неё.
</details>

<details><summary>Задание 3 · про согласие</summary>

**Первый пункт.** Две системы с согласием 0,95 различаются тем, **где** находятся потерянные
5 %. Система A теряет документы с позиций 8–10 — те, что и так почти не влияют на MRR;
потеря конечной метрики около процента. Система B теряет документ с первой позиции в каждом
двадцатом запросе; каждый такой промах обнуляет вклад запроса в MRR почти целиком, и суммарная
потеря доходит до десятков процентов. Согласие у них одинаковое, потому что оно считает
пересечение множеств и слепо к рангу.

**Второй пункт.** Согласие по топ-100 было бы выше при том же индексе: чем шире окно, тем легче
пересечься. Оно стало бы ещё менее чувствительным к тому, что происходит наверху, и потому
для выбора рабочей точки хуже. Общее правило: окно согласия должно совпадать с окном, которое
видит пользователь, — если показываешь десять результатов, меряй согласие@10, а не @100.

**Условие достаточности.** Согласие достаточно, когда промахи равномерно распределены
по позициям. Проверяется гистограммой позиций расхождения между приближённым и точным
поиском — десять строк кода, и вопрос закрыт.
</details>

---

## Литература

* **Malkov & Yashunin (2018), «Efficient and robust approximate nearest neighbor search using
  Hierarchical Navigable Small World graphs»** — первоисточник HNSW. Раздел про `efSearch`
  объясняет, почему жадность приходится страховать очередью кандидатов, а раздел про слои —
  откуда берётся логарифм.
* **Jégou, Douze & Schmid (2011), «Product Quantization for Nearest Neighbor Search»** — вся
  арифметика части 4, включая ADC. Читается тяжелее остальных и стоит того: без ADC непонятно,
  почему PQ вообще быстр, а не только компактен.
* **Johnson, Douze & Jégou (2019), «Billion-scale similarity search with GPUs»** — статья
  про FAISS. Полезна разделом о том, как комбинируют IVF и PQ и почему именно так.
* **Subramanya et al. (2019), «DiskANN»** — как держать индекс на SSD и почему это работает:
  доуточнять надо сотню векторов, а не миллиард.
* **Документация FAISS, «Guidelines to choose an index»** — практическое дерево решений
  на полстраницы. Наш блок «как выбрать индекс за пять минут» — его пересказ с нашими
  акцентами.
* **Aumüller, Bernhardsson & Faithfull, ANN-Benchmarks** — независимое сравнение библиотек
  на одинаковых условиях. Смотреть на кривые «полнота против QPS» целиком, а не на отдельные
  числа: одна точка ничего не доказывает.
* **Лекция L13** и `data/l9-*.json` — числа, с которыми мы сверялись.

**Дальше по курсу.** L15 и неделя 12 поставят поверх выбранной сегодня рабочей точки RAG:
к потере полноты от приближённого поиска добавится потеря от чанкования, и складываться они
будут нелинейно — разбирать это придётся заново, вместе, а не по отдельности.

## Дамп прогона

Правило 10.5: занятие не считается прогнанным, пока его числа не лежат в файле рядом
с конфигурацией рантайма. Ячейка ниже собирает все численные результаты ноутбука —
от сида до финальных метрик — и кладёт их в `runs/lab-ann.json`. Это и есть
доказательство прогона: разбор сверяется с файлом, а не с памятью автора.

In [ ]:
# Дамп прогона — все числовые результаты + конфигурация рантайма (правило 10.5).
import json as _json, os as _os, sys as _sys, platform as _pl, pathlib as _pathlib

_runtime = {"python": _sys.version.split()[0], "platform": _pl.platform()}
_torch = _sys.modules.get("torch")   # НЕ импортируем сами: рамка 7.4 — сид и пин
if _torch is not None:                # обязателен только там, где ноутбук torch ИСПОЛЬЗУЕТ
    _runtime["torch"] = _torch.__version__
    _runtime["gpu"] = _torch.cuda.get_device_name(0) if _torch.cuda.is_available() else None
else:
    import subprocess as _sp
    try:
        _q = _sp.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True, timeout=5)
        _runtime["gpu"] = (_q.stdout.strip().splitlines() or [None])[0] if _q.returncode == 0 else None
    except Exception:
        _runtime["gpu"] = None

def _plain(v):
    try:
        import numpy as _np
        if isinstance(v, _np.integer): return int(v)
        if isinstance(v, _np.floating): return float(v)
    except Exception:
        pass
    return v

def _num(v):
    return isinstance(v, (int, float)) and not isinstance(v, bool)

_metrics = {}
for _k, _v in sorted(globals().items()):
    if _k.startswith("_") or (len(_k) == 1 and _k.islower()):
        continue                       # служебные имена и счётчики циклов
    _v = _plain(_v)
    if _num(_v):
        _metrics[_k] = _v
    elif isinstance(_v, dict) and 0 < len(_v) <= 64 and all(_num(_plain(_x)) for _x in _v.values()):
        _metrics[_k] = {str(_kk): _plain(_vv) for _kk, _vv in _v.items()}
    elif isinstance(_v, (list, tuple)) and 0 < len(_v) <= 64 and all(_num(_plain(_x)) for _x in _v):
        _metrics[_k] = [_plain(_x) for _x in _v]

# Куда писать. Если ядро ВИДИТ репозиторий (локальный запуск, VS Code с локальной ФС,
# смонтированный каталог) — пишем сразу в него, и делать после прогона больше ничего не нужно.
# Прогон на ускорителе кладётся отдельно в seminars/runs-t4/, чтобы не затирать базовый
# CPU-дамп: гейт G29 сверяет эти два прогона между собой (качество строго, тайминги с допуском).
def _repo_runs():
    _here = _pathlib.Path.cwd().resolve()
    for _cand in [_here] + list(_here.parents):
        if (_cand / "data" / "course.json").exists() and (_cand / "seminars").is_dir():
            return _cand / "seminars" / ("runs-t4" if _runtime.get("gpu") else "runs")
    return None

if _os.environ.get("RUNS_DIR"):
    _out = _pathlib.Path(_os.environ["RUNS_DIR"])
else:
    _out = _repo_runs() or _pathlib.Path("runs")   # в песочнице Colab репозитория нет
_out.mkdir(parents=True, exist_ok=True)
_path = _out / "lab-ann.json"
_payload = {"notebook": "lab-ann", "runtime": _runtime, "metrics": _metrics}
_json.dump(_payload, open(_path, "w", encoding="utf-8"),
           ensure_ascii=False, indent=1, sort_keys=True)
print(f"дамп: {_path} · величин: {len(_metrics)} · рантайм: {_runtime['gpu'] or 'CPU'}")

# И ПЕЧАТАЕМ дамп между маркерами — на случай, когда файл выше остался в песочнице
# Colab и до репозитория не доехал: вывод ячейки сохраняется прямо в .ipynb. Верни
# тетрадку в репозиторий (или просто скачай её) — наблюдатель `npm run runs:watch`
# достанет дамп из вывода сам; вручную то же делает `npm run runs`. Скачивать
# отдельные JSON-файлы руками не нужно ни в одном из случаев.
print("<<<DLS-RUN-DUMP")
print(_json.dumps(_payload, ensure_ascii=False, sort_keys=True))
print("DLS-RUN-DUMP>>>")

**Что видно.** В дампе — конфигурация прогона и все скалярные результаты по именам
переменных. Сравнивать надо не тайминги — они свойство рантайма, и на T4, A100 и CPU
законно разные, — а метрики качества: при одном сиде они обязаны совпасть до знака.
Если твой прогон разошёлся с эталонным в качестве, а не во времени, — это находка,
неси её на занятие.